# 1. Dataset Discovery and Initial Validation

Locate the mounted dataset and establish its structure before training.

In [ ]:

# Cell 2: Inspect dataset folder

import os

DATASET_DIR = "/kaggle/input/dataset"

print("Dataset path exists:", os.path.exists(DATASET_DIR))

for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, "").count(os.sep)

    print(f"\n{'  ' * level}{os.path.basename(root)}/")

    for filename in files[:15]:
        print(f"{'  ' * (level + 1)}{filename}")

    if level >= 3:
        dirs[:] = []

print("\nInspection finished.")



In [ ]:

# Cell 3: Locate and inspect the mounted dataset

import os


BASE_DIR = "/kaggle/input/datasets/ajithdari"

for root, dirs, files in os.walk(BASE_DIR):
    level = root.replace(BASE_DIR, "").count(os.sep)

    print("  " * level + os.path.basename(root) + "/")

    for filename in files[:20]:
        print("  " * (level + 1) + filename)

    # Keep the initial inspection short
    if level >= 3:
        dirs[:] = []

print("\nInspection complete.")



# 2. Dataset and Label Integrity Audits

Validate CSV schema, paths, labels, duplicates, and split integrity before modeling.

In [ ]:

# Cell 4: Inspect CSV schema and image counts

import os
import pandas as pd

DATASET_DIR = (
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset"
)

for split in ["train", "valid", "test"]:
    csv_path = os.path.join(DATASET_DIR, "Dataset_csv", f"{split}.csv")
    image_dir = os.path.join(DATASET_DIR, "Dataset_img", split)

    print(f"\n{'=' * 50}")
    print(f"SPLIT: {split.upper()}")

    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        print("CSV shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print("\nFirst 3 rows:")
        print(df.head(3).to_string(index=False))
    else:
        print("CSV missing:", csv_path)

    if os.path.isdir(image_dir):
        image_files = [
            f for f in os.listdir(image_dir)
            if os.path.isfile(os.path.join(image_dir, f))
        ]
        print("\nImages directly in folder:", len(image_files))
        print("Sample filenames:", image_files[:10])
    else:
        print("Image folder missing:", image_dir)

print("\nMetadata inspection complete.")


In [ ]:

# Cell 5: Verify CSV image paths

import os
import pandas as pd

DATASET_DIR = (
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset"
)

for split in ["train", "valid", "test"]:
    csv_path = os.path.join(DATASET_DIR, "Dataset_csv", f"{split}.csv")
    df = pd.read_csv(csv_path)

    print(f"\n{'=' * 45}")
    print(split.upper(), "rows:", len(df))

    # Check the original paths embedded in the CSV
    exists_original = df["image_path"].astype(str).apply(os.path.isfile)
    print("Original CSV paths that exist:", int(exists_original.sum()))

    # Inspect actual images under this split
    image_root = os.path.join(DATASET_DIR, "Dataset_img", split)
    image_files = []

    for root, dirs, files in os.walk(image_root):
        for filename in files:
            if filename.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp", ".webp")
            ):
                image_files.append(os.path.join(root, filename))

    print("Image files found recursively:", len(image_files))
    print("Sample actual image paths:", image_files[:5])

print("\nPath verification complete. No files modified.")


In [ ]:

# Cell 6: Audit CSV-to-image correspondence

import os
import pandas as pd
from collections import Counter

DATASET_DIR = (
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset"
)

for split in ["train", "valid", "test"]:
    df = pd.read_csv(
        os.path.join(DATASET_DIR, "Dataset_csv", f"{split}.csv")
    )

    image_root = os.path.join(DATASET_DIR, "Dataset_img", split)

    # Count images by class folder
    folder_counts = {}
    for label_folder in ["autistic", "non_autistic", "non-autistic", "typical"]:
        folder = os.path.join(image_root, label_folder)
        if os.path.isdir(folder):
            folder_counts[label_folder] = len([
                f for f in os.listdir(folder)
                if f.lower().endswith((".jpg", ".jpeg", ".png"))
            ])

    print(f"\n--- {split.upper()} ---")
    print("CSV rows:", len(df))
    print("CSV Class counts:")
    print(df["Class"].astype(str).str.strip().str.upper().value_counts().to_dict())
    print("Image counts by folder:", folder_counts)

    # Show how the first rows' stale filenames compare with actual files
    print("\nFirst 5 CSV image paths:")
    print(df["image_path"].head().tolist())

    print("\nFirst 5 actual image names per class folder:")
    for folder_name in folder_counts:
        folder = os.path.join(image_root, folder_name)
        names = sorted(os.listdir(folder))[:5]
        print(folder_name, names)

print("\nAudit only; no files modified.")


In [ ]:

# Cell 7: Verify labels against image folders

import os
import pandas as pd

DATASET_DIR = (
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset"
)

for split in ["train", "valid", "test"]:
    csv_path = os.path.join(DATASET_DIR, "Dataset_csv", f"{split}.csv")
    image_root = os.path.join(DATASET_DIR, "Dataset_img", split)
    df = pd.read_csv(csv_path)

    # Map CSV class labels to expected folder names
    expected_folder = (
        df["Class"].astype(str).str.strip().str.upper()
        .map({"YES": "autistic", "NO": "non_autistic"})
    )

    # Count actual images in each class folder
    actual_counts = {}
    for folder_name in ["autistic", "non_autistic"]:
        folder = os.path.join(image_root, folder_name)
        actual_counts[folder_name] = len([
            f for f in os.listdir(folder)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

    print(f"\n{split.upper()}")
    print("CSV rows:", len(df))
    print("Unknown CSV class labels:", int(expected_folder.isna().sum()))
    print("Image counts:", actual_counts)
    print("CSV label counts:",
          df["Class"].astype(str).str.strip().str.upper().value_counts().to_dict())

print("\nThis checks class counts only, not individual row-to-image pairing.")


In [ ]:

# Cell 8: Inspect questionnaire patterns and original filenames

import os
import pandas as pd

DATASET_DIR = (
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset"
)

df = pd.read_csv(os.path.join(DATASET_DIR, "Dataset_csv", "train.csv"))

print("Columns and data types:")
print(df.dtypes.to_string())

print("\nQuestionnaire and demographic sample:")
print(df[["A1", "A2", "A3", "A4", "A5",
          "A6", "A7", "A8", "A9", "A10",
          "Age", "Sex", "Jauundice", "Family_ASD", "Class"]]
      .head(10).to_string(index=False))

print("\nQuestionnaire answer totals by class:")
items = [f"A{i}" for i in range(1, 11)]
print(df.groupby("Class")[items].agg(["mean", "sum"]).round(3))

print("\nOriginal image filename examples:")
print(df["image_path"].astype(str).head(10).tolist())

print("\nThis is a provenance check, not proof of individual pairing.")


In [ ]:

# Cell 9: Questionnaire duplication and label audit

import os
import pandas as pd

DATASET_DIR = (
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset"
)

items = [f"A{i}" for i in range(1, 11)]

for split in ["train", "valid", "test"]:
    df = pd.read_csv(
        os.path.join(DATASET_DIR, "Dataset_csv", f"{split}.csv")
    )

    # Count repeated answer patterns
    repeated_patterns = df.duplicated(subset=items, keep=False)

    # Check whether identical answer patterns have conflicting labels
    label_counts = df.groupby(items)["Class"].nunique()
    conflicting_patterns = int((label_counts > 1).sum())

    print(f"\n--- {split.upper()} ---")
    print("Rows:", len(df))
    print("Unique A1-A10 patterns:", df[items].drop_duplicates().shape[0])
    print("Rows belonging to repeated patterns:", int(repeated_patterns.sum()))
    print("Patterns associated with multiple labels:", conflicting_patterns)
    print("\nClass counts:")
    print(df["Class"].value_counts().to_dict())

print("\nAudit complete; no files modified.")


In [ ]:

# Cell 10: Cross-split questionnaire pattern overlap

import os
import pandas as pd


DATASET_DIR = (
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset"
)

items = [f"A{i}" for i in range(1, 11)]
data = {}

for split in ["train", "valid", "test"]:
    data[split] = pd.read_csv(
        os.path.join(DATASET_DIR, "Dataset_csv", f"{split}.csv")
    )

patterns = {
    split: set(map(tuple, df[items].to_numpy()))
    for split, df in data.items()
}

for a, b in [("train", "valid"), ("train", "test"), ("valid", "test")]:
    shared = patterns[a] & patterns[b]
    print(f"{a} vs {b}:")
    print("  Unique patterns in first split:", len(patterns[a]))
    print("  Unique patterns in second split:", len(patterns[b]))
    print("  Shared unique patterns:", len(shared))

print("\nNo files modified.")


# 3. Image-Only Baseline: MobileNetV2

Train/evaluate the image-only baseline. Use saved checkpoints where available.

In [ ]:

# Cell 18: MobileNetV2 image-only baseline
# Training cell — uses GPU when available

import os

import json
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

SEED = 42
tf.keras.utils.set_random_seed(SEED)

DATASET_DIR = (
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset"
)
OUT_DIR = "/kaggle/working/asd_care_results"
os.makedirs(OUT_DIR, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

def load_split(split):
    csv_path = os.path.join(
        DATASET_DIR, "Dataset_csv", f"{split}.csv"
    )
    df = pd.read_csv(csv_path)

    paths, labels_list = [], []

    for _, row in df.iterrows():
        label = str(row["Class"]).strip().upper()
        folder = "autistic" if label == "YES" else "non_autistic"
        path = os.path.join(
            DATASET_DIR, "Dataset_img", split,
            folder, os.path.basename(str(row["image_path"]))
        )

        if not os.path.isfile(path):
            raise FileNotFoundError(path)

        paths.append(path)
        labels_list.append(1 if label == "YES" else 0)

    return np.array(paths), np.array(labels_list, dtype=np.float32)

def make_dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(
            len(paths), seed=SEED, reshuffle_each_iteration=True
        )

    def load_image(path, label):
        image_bytes = tf.io.read_file(path)
        image = tf.io.decode_image(
            image_bytes, channels=3, expand_animations=False
        )
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMG_SIZE)
        image = preprocess_input(tf.cast(image, tf.float32))
        return image, label

    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_paths, y_train = load_split("train")
valid_paths, y_valid = load_split("valid")
test_paths, y_test = load_split("test")

train_ds = make_dataset(train_paths, y_train, training=True)
valid_ds = make_dataset(valid_paths, y_valid)
test_ds = make_dataset(test_paths, y_test)

print("Split sizes:", len(train_paths), len(valid_paths), len(test_paths))
print("Label counts — train:", np.unique(y_train, return_counts=True))
print("Label counts — valid:", np.unique(y_valid, return_counts=True))
print("Label counts — test:", np.unique(y_test, return_counts=True))

# Skip training if a completed best-model artifact already exists.
best_model_path = os.path.join(OUT_DIR, "mobilenetv2_best.keras")
history_path = os.path.join(OUT_DIR, "mobilenetv2_history.json")

if os.path.exists(best_model_path) and os.path.exists(history_path):
    print("\nExisting model and history found. No retraining performed.")
    print("Model:", best_model_path)
else:
    base = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base.trainable = False

    inputs = layers.Input(shape=(224, 224, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = Model(inputs, outputs, name="mobilenetv2_image_only")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            best_model_path,
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=3,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.CSVLogger(
            os.path.join(OUT_DIR, "mobilenetv2_training_log.csv")
        ),
    ]

    history = model.fit(
        train_ds,
        validation_data=valid_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )

    with open(history_path, "w") as f:
        json.dump(
            {k: [float(v) for v in values]
             for k, values in history.history.items()},
            f,
            indent=2,
        )

    print("\nTraining complete.")
    print("Best model:", best_model_path)
    print("History:", history_path)


In [ ]:

# Cell 19: Final test evaluation — MobileNetV2
# Loads the saved checkpoint; does not retrain

import os
import json
import numpy as np
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

OUT_DIR = "/kaggle/working/asd_care_results"
MODEL_PATH = os.path.join(OUT_DIR, "mobilenetv2_best.keras")

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(f"Saved model not found: {MODEL_PATH}")

best_model = tf.keras.models.load_model(MODEL_PATH)

# Use the test dataset created in Cell 18.
y_true = np.asarray(y_test).astype(int)
y_prob = best_model.predict(test_ds, verbose=1).reshape(-1)
y_pred = (y_prob >= 0.5).astype(int)

results = {
    "model": "MobileNetV2 image-only",
    "test_samples": int(len(y_true)),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "precision": float(precision_score(y_true, y_pred, zero_division=0)),
    "recall": float(recall_score(y_true, y_pred, zero_division=0)),
    "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_true, y_prob)),
    "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
}

print("\nTEST RESULTS")
print(json.dumps(results, indent=2))
print("\nClassification report:")
print(classification_report(
    y_true, y_pred,
    target_names=["NO", "YES"],
    zero_division=0
))

# Save results separately; preserve all existing artifacts.
result_path = os.path.join(OUT_DIR, "mobilenetv2_test_results.json")
with open(result_path, "w") as f:
    json.dump(results, f, indent=2)

np.savez_compressed(
    os.path.join(OUT_DIR, "mobilenetv2_test_predictions.npz"),
    y_true=y_true,
    y_prob=y_prob,
    y_pred=y_pred,
)

print("\nSaved:", result_path)
print("Saved test predictions.")


In [ ]:

# Cell 20B: FIXED questionnaire-only model
# Correctly encodes categorical features.
# Does NOT overwrite MobileNetV2 artifacts.

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

DATA_DIR = (
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset/Dataset_csv"
)
OUT_DIR = "/kaggle/working/asd_care_results"
os.makedirs(OUT_DIR, exist_ok=True)

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
valid_df = pd.read_csv(os.path.join(DATA_DIR, "valid.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

NUMERIC_FEATURES = [
    "A1", "A2", "A3", "A4", "A5",
    "A6", "A7", "A8", "A9", "A10", "Age"
]
CATEGORICAL_FEATURES = ["Sex", "Jauundice", "Family_ASD"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "Class"

# Confirm expected columns.
for name, df in [
    ("train", train_df), ("valid", valid_df), ("test", test_df)
]:
    missing = [c for c in FEATURES + [TARGET] if c not in df.columns]
    if missing:
        raise ValueError(f"{name}.csv missing columns: {missing}")

# Convert genuine numeric columns only.
for df in [train_df, valid_df, test_df]:
    for col in NUMERIC_FEATURES:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Impute numeric missing values using training medians only.
numeric_medians = train_df[NUMERIC_FEATURES].median()
if numeric_medians.isna().any():
    raise ValueError(
        "At least one numeric feature is entirely missing in training data."
    )

for df in [train_df, valid_df, test_df]:
    df[NUMERIC_FEATURES] = df[NUMERIC_FEATURES].fillna(numeric_medians)

# Normalize numerical features using training data only.
scaler = StandardScaler()
X_train_num = scaler.fit_transform(
    train_df[NUMERIC_FEATURES]
).astype("float32")
X_valid_num = scaler.transform(
    valid_df[NUMERIC_FEATURES]
).astype("float32")
X_test_num = scaler.transform(
    test_df[NUMERIC_FEATURES]
).astype("float32")

# One-hot encode categories using training categories only.
train_cat = train_df[CATEGORICAL_FEATURES].fillna("unknown").astype(str)
valid_cat = valid_df[CATEGORICAL_FEATURES].fillna("unknown").astype(str)
test_cat  = test_df[CATEGORICAL_FEATURES].fillna("unknown").astype(str)

cat_columns = []
category_levels = {}

for col in CATEGORICAL_FEATURES:
    levels = sorted(train_cat[col].unique().tolist())
    category_levels[col] = levels

    # Fixed columns ensure identical feature layout for every split.
    for level in levels:
        safe_level = level.replace(" ", "_")
        new_col = f"{col}__{safe_level}"
        cat_columns.append((col, level, new_col))

def encode_categories(frame):
    encoded = []
    for col, level, new_col in cat_columns:
        encoded.append(
            (frame[col] == level).to_numpy(dtype="float32")
        )
    return np.column_stack(encoded).astype("float32")

X_train_cat = encode_categories(train_cat)
X_valid_cat = encode_categories(valid_cat)
X_test_cat  = encode_categories(test_cat)

X_train = np.concatenate([X_train_num, X_train_cat], axis=1)
X_valid = np.concatenate([X_valid_num, X_valid_cat], axis=1)
X_test  = np.concatenate([X_test_num, X_test_cat], axis=1)

def encode_labels(series):
    s = series.astype(str).str.strip().str.upper()
    mapping = {
        "YES": 1, "NO": 0,
        "ASD": 1, "TD": 0,
        "AUTISTIC": 1, "NON_AUTISTIC": 0,
        "NON-AUTISTIC": 0,
        "TRUE": 1, "FALSE": 0,
        "1": 1, "0": 0
    }
    result = s.map(mapping)
    if result.isna().any():
        raise ValueError(
            f"Unexpected labels: {sorted(s[result.isna()].unique())}"
        )
    return result.to_numpy(dtype="int32")

y_train = encode_labels(train_df[TARGET])
y_valid = encode_labels(valid_df[TARGET])
y_test_q = encode_labels(test_df[TARGET])

# Safety checks before training.
for name, X in [
    ("train", X_train), ("validation", X_valid), ("test", X_test)
]:
    if not np.isfinite(X).all():
        raise ValueError(f"{name} features still contain NaN or infinity.")

print("Train:", X_train.shape, "labels:", np.bincount(y_train))
print("Valid:", X_valid.shape, "labels:", np.bincount(y_valid))
print("Test: ", X_test.shape, "labels:", np.bincount(y_test_q))
print("All feature matrices are finite.")

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(1, activation="sigmoid")
], name="questionnaire_only_mlp")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

MODEL_PATH = os.path.join(OUT_DIR, "questionnaire_mlp_best.keras")

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_PATH, monitor="val_auc", mode="max",
        save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc", mode="max",
        patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        os.path.join(OUT_DIR, "questionnaire_mlp_training_log.csv")
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

with open(os.path.join(OUT_DIR, "questionnaire_mlp_history.json"), "w") as f:
    json.dump(
        {k: [float(v) for v in vals]
         for k, vals in history.history.items()},
        f, indent=2
    )

with open(os.path.join(OUT_DIR, "questionnaire_preprocessing.json"), "w") as f:
    json.dump({
        "numeric_features": NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "category_levels": category_levels,
        "numeric_medians": numeric_medians.to_dict(),
        "scaler_mean": scaler.mean_.tolist(),
        "scaler_scale": scaler.scale_.tolist(),
        "encoded_feature_count": int(X_train.shape[1])
    }, f, indent=2)

print("\nQuestionnaire training complete.")
print("Best model saved:", MODEL_PATH)


# 5. Checkpoints, Recovery, and Experiment Tracking

Inspect saved artifacts and ensure the experiment can resume without unnecessary retraining.

In [ ]:

# Cell 24: Locate saved experiment files
import os

targets = {
    "mobilenetv2_best.keras",
    "mobilenetv2_test_results.json",
    "mobilenetv2_test_predictions.npz",
    "questionnaire_mlp_best.keras",
    "questionnaire_mlp_test_results.json",
    "questionnaire_mlp_test_predictions.npz",
    "efficientnetb0_best.keras",
}

roots = [
    "/kaggle/working",
    "/kaggle/input/datasets/shahbaz18/asd-full-backup",
    "/kaggle/input/datasets/ajithdari/autism-spectrum-disorder-multi-modal-dataset",
]

found = []
for root in roots:
    if not os.path.isdir(root):
        print("PATH NOT AVAILABLE:", root)
        continue

    print("\nSearching:", root)
    for current, dirs, files in os.walk(root):
        # Avoid descending into unrelated hidden folders
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        for name in files:
            if name in targets or (
                name.endswith(".keras") and
                any(key in name.lower() for key in
                    ["mobilenet", "questionnaire", "efficientnet"])
            ):
                path = os.path.join(current, name)
                found.append(path)
                print("FOUND:", path)

if not found:
    print("\nNo matching files found in these locations.")
else:
    print(f"\nTotal matching files: {len(found)}")


In [ ]:

# Cell: Verify checkpoints and create a recovery archive
import os
import zipfile

out = "/kaggle/working/asd_care_results"
os.makedirs(out, exist_ok=True)

required = [
    "mobilenetv2_best.keras",
    "questionnaire_mlp_best.keras",
    "mobilenetv2_test_predictions.npz",
    "mobilenetv2_test_results.json",
]

print("CHECKPOINT VERIFICATION")
for name in required:
    path = os.path.join(out, name)
    exists = os.path.isfile(path)
    size_mb = os.path.getsize(path) / (1024 * 1024) if exists else 0
    print(f"{'FOUND' if exists else 'MISSING'}: {name} ({size_mb:.2f} MB)")

archive = "/kaggle/working/asd_care_recovery.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(out):
        for name in files:
            path = os.path.join(root, name)
            z.write(path, arcname=os.path.relpath(path, "/kaggle/working"))

print("\nRecovery ZIP:", archive)
print("ZIP size (MB):", round(os.path.getsize(archive) / (1024 * 1024), 2))



In [ ]:

# Cell: Rebuild Questionnaire MLP Test Results — no training
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

out = "/kaggle/working/asd_care_results"
csv_path = (
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset/"
    "Dataset_csv/test.csv"
)

# Load saved preprocessing configuration and test data
with open(os.path.join(out, "questionnaire_preprocessing.json")) as f:
    prep = json.load(f)

df = pd.read_csv(csv_path)

numeric = prep["numeric_features"]
categorical = prep["categorical_features"]

# Recreate numeric preprocessing
num = df[numeric].apply(pd.to_numeric, errors="coerce")
for col in numeric:
    num[col] = num[col].fillna(prep["numeric_medians"][col])

means = np.array(prep["scaler_mean"], dtype=float)
scales = np.array(prep["scaler_scale"], dtype=float)
X_num = (num.to_numpy(dtype=float) - means) / scales

# Recreate categorical one-hot encoding in saved order
cat_parts = []
for col in categorical:
    values = df[col].astype(str).str.strip().str.lower()
    levels = prep["category_levels"][col]
    encoded = np.column_stack([
        (values == level).astype(np.float32)
        for level in levels
    ])
    cat_parts.append(encoded)

X = np.concatenate([X_num, *cat_parts], axis=1).astype(np.float32)

# Validate before inference
assert X.shape[1] == prep["encoded_feature_count"] == 17, X.shape
assert len(X) == len(df)

y_true = (df["Class"].astype(str).str.strip().str.upper() == "YES").astype(int).to_numpy()

# Load checkpoint and predict
model = tf.keras.models.load_model(
    os.path.join(out, "questionnaire_mlp_best.keras")
)
y_prob = model.predict(X, verbose=0).reshape(-1)
y_pred = (y_prob >= 0.5).astype(int)

results = {
    "model": "Questionnaire-only MLP",
    "test_samples": int(len(y_true)),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "precision": float(precision_score(y_true, y_pred, zero_division=0)),
    "recall": float(recall_score(y_true, y_pred, zero_division=0)),
    "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_true, y_prob)),
    "confusion_matrix": confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).tolist(),
    "threshold": 0.5,
    "note": "Re-evaluated from saved checkpoint and reconstructed preprocessing."
}

with open(os.path.join(out, "questionnaire_mlp_test_results.json"), "w") as f:
    json.dump(results, f, indent=2)

np.savez_compressed(
    os.path.join(out, "questionnaire_mlp_test_predictions.npz"),
    y_true=y_true,
    y_prob=y_prob,
    y_pred=y_pred
)

print(json.dumps(results, indent=2))
print("\nSaved questionnaire test results and predictions.")


In [ ]:

# Cell: Backup all current experiment results
import os
import zipfile

out = "/kaggle/working/asd_care_results"
archive = "/kaggle/working/asd_care_recovery.zip"

assert os.path.isfile(os.path.join(out, "mobilenetv2_best.keras"))
assert os.path.isfile(os.path.join(out, "questionnaire_mlp_best.keras"))
assert os.path.isfile(os.path.join(out, "questionnaire_mlp_test_results.json"))
assert os.path.isfile(os.path.join(out, "questionnaire_mlp_test_predictions.npz"))

with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(out):
        for name in files:
            path = os.path.join(root, name)
            z.write(path, os.path.relpath(path, "/kaggle/working"))

print("Backup created:", archive)
print("Size (MB):", round(os.path.getsize(archive) / (1024 * 1024), 2))
print("\nFiles included:")
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        print(name)


# 6. Additional Image Models: EfficientNetB0 and ResNet50

Run only when the required checkpoints/results are missing; evaluation cells use the held-out test set.

In [ ]:

# Cell: EfficientNetB0 Resume Check — no training
import os
import tensorflow as tf

out = "/kaggle/working/asd_care_results"

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

print("\nExisting EfficientNetB0 files:")
for name in [
    "efficientnetb0_best.keras",
    "efficientnetb0_history.json",
    "efficientnetb0_training_log.csv",
    "efficientnetb0_test_results.json",
    "efficientnetb0_test_predictions.npz",
]:
    path = os.path.join(out, name)
    print(("FOUND   " if os.path.isfile(path) else "MISSING "), name)

print("\nDataset split files:")
base = (
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset"
)
for name in ["train.csv", "valid.csv", "test.csv"]:
    path = os.path.join(base, "Dataset_csv", name)
    print(("FOUND   " if os.path.isfile(path) else "MISSING "), path)


In [ ]:

# Cell: EfficientNetB0 Dataset Preflight — no training
import os
import pandas as pd

base = (
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset"
)
csv_dir = os.path.join(base, "Dataset_csv")

for split in ["train", "valid", "test"]:
    path = os.path.join(csv_dir, f"{split}.csv")
    df = pd.read_csv(path)

    print(f"\n=== {split.upper()} ===")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    print("Class counts:")
    print(df["Class"].value_counts(dropna=False).to_dict())

    # Validate image paths without loading the images
    missing = []
    for p in df["image_path"].dropna().astype(str):
        if not os.path.isfile(p):
            missing.append(p)

    print("Missing image paths:", len(missing))
    if missing:
        print("Examples:", missing[:5])

print("\nPreflight complete. No model trained.")


In [ ]:

# Cell: Train EfficientNetB0 — checkpointed and resumable
import os
import json
import tensorflow as tf
import pandas as pd
import numpy as np

from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, CSVLogger
)

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

tf.keras.utils.set_random_seed(SEED)

out = "/kaggle/working/asd_care_results"
os.makedirs(out, exist_ok=True)

base = (
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset"
)
csv_dir = os.path.join(base, "Dataset_csv")

# Load split manifests
train_df = pd.read_csv(os.path.join(csv_dir, "train.csv"))
valid_df = pd.read_csv(os.path.join(csv_dir, "valid.csv"))
test_df  = pd.read_csv(os.path.join(csv_dir, "test.csv"))

def make_dataset(df, training=False):
    paths = df["image_path"].astype(str).to_numpy()
    labels = (
        df["Class"].astype(str).str.strip().str.upper() == "YES"
    ).astype(np.float32).to_numpy()

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def load_image(path, label):
        image_bytes = tf.io.read_file(path)
        image = tf.io.decode_image(
            image_bytes, channels=3, expand_animations=False
        )
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMG_SIZE)
        image = tf.cast(image, tf.float32)
        return image, label

    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
valid_ds = make_dataset(valid_df)
test_ds = make_dataset(test_df)

# Build model
backbone = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
backbone.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = backbone(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = Model(inputs, outputs, name="EfficientNetB0_ASD_image_only")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc")
    ]
)

checkpoint = os.path.join(out, "efficientnetb0_best.keras")
history_path = os.path.join(out, "efficientnetb0_history.json")
log_path = os.path.join(out, "efficientnetb0_training_log.csv")

callbacks = [
    ModelCheckpoint(
        checkpoint, monitor="val_auc", mode="max",
        save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor="val_auc", mode="max",
        patience=3, restore_best_weights=True, verbose=1
    ),
    CSVLogger(log_path, append=True)
]

print("Train:", len(train_df), "| Validation:", len(valid_df))
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Starting EfficientNetB0 training...")

hist = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

with open(history_path, "w") as f:
    json.dump(
        {k: [float(v) for v in values]
         for k, values in hist.history.items()},
        f, indent=2
    )

print("\nTraining finished.")
print("Best checkpoint exists:", os.path.isfile(checkpoint))
print("History saved:", os.path.isfile(history_path))
print("Training log saved:", os.path.isfile(log_path))


In [ ]:

# Cell: Evaluate EfficientNetB0 on the held-out test set
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

out = "/kaggle/working/asd_care_results"
base = (
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset"
)

test_df = pd.read_csv(os.path.join(base, "Dataset_csv", "test.csv"))
checkpoint = os.path.join(out, "efficientnetb0_best.keras")

assert os.path.isfile(checkpoint), "EfficientNetB0 checkpoint missing."

# Recreate test dataset with the same image processing
paths = test_df["image_path"].astype(str).to_numpy()
y_true = (
    test_df["Class"].astype(str).str.strip().str.upper() == "YES"
).astype(int).to_numpy()

def load_test_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, (224, 224))
    return tf.cast(image, tf.float32), label

test_ds = tf.data.Dataset.from_tensor_slices(
    (paths, y_true.astype(np.float32))
)
test_ds = test_ds.map(
    load_test_image, num_parallel_calls=tf.data.AUTOTUNE
).batch(32).prefetch(tf.data.AUTOTUNE)

# Load the saved best checkpoint; no training
model = tf.keras.models.load_model(checkpoint)
y_prob = model.predict(test_ds, verbose=1).reshape(-1)
y_pred = (y_prob >= 0.5).astype(int)

results = {
    "model": "EfficientNetB0 image-only",
    "test_samples": int(len(y_true)),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "precision": float(precision_score(y_true, y_pred, zero_division=0)),
    "recall": float(recall_score(y_true, y_pred, zero_division=0)),
    "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_true, y_prob)),
    "confusion_matrix": confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).tolist(),
    "threshold": 0.5
}

with open(os.path.join(out, "efficientnetb0_test_results.json"), "w") as f:
    json.dump(results, f, indent=2)

np.savez_compressed(
    os.path.join(out, "efficientnetb0_test_predictions.npz"),
    y_true=y_true, y_prob=y_prob, y_pred=y_pred
)

print("\nTEST RESULTS")
print(json.dumps(results, indent=2))
print("\nSaved test metrics and predictions.")


In [ ]:

# Cell: Train ResNet50 — checkpointed, no duplicate training
import os
import json
import tensorflow as tf
import pandas as pd
import numpy as np

from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, CSVLogger

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

tf.keras.utils.set_random_seed(SEED)

out = "/kaggle/working/asd_care_results"
os.makedirs(out, exist_ok=True)

checkpoint = os.path.join(out, "resnet50_best.keras")
history_path = os.path.join(out, "resnet50_history.json")
log_path = os.path.join(out, "resnet50_training_log.csv")

if os.path.isfile(checkpoint):
    print("ResNet50 checkpoint already exists — skipping training.")
    print("Checkpoint:", checkpoint)
else:
    base = (
        "/kaggle/input/datasets/ajithdari/"
        "autism-spectrum-disorder-multi-modal-dataset"
    )
    csv_dir = os.path.join(base, "Dataset_csv")
    train_df = pd.read_csv(os.path.join(csv_dir, "train.csv"))
    valid_df = pd.read_csv(os.path.join(csv_dir, "valid.csv"))

    def make_dataset(df, training=False):
        paths = df["image_path"].astype(str).to_numpy()
        labels = (
            df["Class"].astype(str).str.strip().str.upper() == "YES"
        ).astype(np.float32).to_numpy()

        ds = tf.data.Dataset.from_tensor_slices((paths, labels))

        def load_image(path, label):
            image = tf.io.read_file(path)
            image = tf.io.decode_image(
                image, channels=3, expand_animations=False
            )
            image.set_shape([None, None, 3])
            image = tf.image.resize(image, IMG_SIZE)
            image = preprocess_input(tf.cast(image, tf.float32))
            return image, label

        ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
        if training:
            ds = ds.shuffle(
                len(df), seed=SEED, reshuffle_each_iteration=True
            )
        return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    train_ds = make_dataset(train_df, training=True)
    valid_ds = make_dataset(valid_df)

    backbone = ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3)
    )
    backbone.trainable = False

    inputs = layers.Input(shape=(224, 224, 3))
    x = backbone(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = Model(inputs, outputs, name="ResNet50_ASD_image_only")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )

    callbacks = [
        ModelCheckpoint(
            checkpoint, monitor="val_auc", mode="max",
            save_best_only=True, verbose=1
        ),
        EarlyStopping(
            monitor="val_auc", mode="max",
            patience=3, restore_best_weights=True, verbose=1
        ),
        CSVLogger(log_path, append=False)
    ]

    print("Train rows:", len(train_df))
    print("Validation rows:", len(valid_df))
    print("GPUs:", tf.config.list_physical_devices("GPU"))
    print("Starting ResNet50 training...")

    hist = model.fit(
        train_ds,
        validation_data=valid_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    with open(history_path, "w") as f:
        json.dump(
            {k: [float(v) for v in values]
             for k, values in hist.history.items()},
            f, indent=2
        )

    print("\nTraining complete.")
    print("Checkpoint saved:", os.path.isfile(checkpoint))
    print("History saved:", os.path.isfile(history_path))
    print("Training log saved:", os.path.isfile(log_path))


In [ ]:

# Cell: Evaluate ResNet50 on held-out test data
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

out = "/kaggle/working/asd_care_results"
base = (
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset"
)

test_df = pd.read_csv(os.path.join(base, "Dataset_csv", "test.csv"))
checkpoint = os.path.join(out, "resnet50_best.keras")
assert os.path.isfile(checkpoint), "ResNet50 checkpoint missing."

paths = test_df["image_path"].astype(str).to_numpy()
y_true = (
    test_df["Class"].astype(str).str.strip().str.upper() == "YES"
).astype(int).to_numpy()

def load_test_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, (224, 224))
    image = preprocess_input(tf.cast(image, tf.float32))
    return image, label

test_ds = tf.data.Dataset.from_tensor_slices(
    (paths, y_true.astype(np.float32))
)
test_ds = test_ds.map(
    load_test_image, num_parallel_calls=tf.data.AUTOTUNE
).batch(32).prefetch(tf.data.AUTOTUNE)

model = tf.keras.models.load_model(checkpoint)
y_prob = model.predict(test_ds, verbose=1).reshape(-1)
y_pred = (y_prob >= 0.5).astype(int)

results = {
    "model": "ResNet50 image-only",
    "test_samples": int(len(y_true)),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "precision": float(precision_score(y_true, y_pred, zero_division=0)),
    "recall": float(recall_score(y_true, y_pred, zero_division=0)),
    "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_true, y_prob)),
    "confusion_matrix": confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).tolist(),
    "threshold": 0.5
}

with open(os.path.join(out, "resnet50_test_results.json"), "w") as f:
    json.dump(results, f, indent=2)

np.savez_compressed(
    os.path.join(out, "resnet50_test_predictions.npz"),
    y_true=y_true, y_prob=y_prob, y_pred=y_pred
)

print("\nTEST RESULTS")
print(json.dumps(results, indent=2))
print("\nSaved ResNet50 test metrics and predictions.")


# 7. Ablation Results and Multimodal Pairing Audit

Compare model branches and explicitly audit whether image and questionnaire records are truly linked.

In [ ]:

# Cell: Final Ablation Comparison + Recovery Backup
import os
import json
import zipfile
import pandas as pd

out = "/kaggle/working/asd_care_results"
os.makedirs(out, exist_ok=True)

models = {
    "MobileNetV2": "mobilenetv2_test_results.json",
    "Questionnaire MLP": "questionnaire_mlp_test_results.json",
    "EfficientNetB0": "efficientnetb0_test_results.json",
    "ResNet50": "resnet50_test_results.json",
}

rows = []
missing = []

for model_name, filename in models.items():
    path = os.path.join(out, filename)
    if not os.path.isfile(path):
        missing.append(filename)
        continue

    with open(path, "r") as f:
        result = json.load(f)

    rows.append({
        "model": model_name,
        "test_samples": result.get("test_samples"),
        "accuracy": result.get("accuracy"),
        "precision": result.get("precision"),
        "recall": result.get("recall"),
        "f1": result.get("f1"),
        "roc_auc": result.get("roc_auc"),
        "confusion_matrix": json.dumps(result.get("confusion_matrix"))
    })

if missing:
    print("Missing results:", missing)

comparison = pd.DataFrame(rows)

if not comparison.empty:
    comparison = comparison.sort_values("roc_auc", ascending=False)

    csv_path = os.path.join(out, "final_model_comparison.csv")
    json_path = os.path.join(out, "final_model_comparison.json")

    comparison.to_csv(csv_path, index=False)
    comparison.to_json(json_path, orient="records", indent=2)

    print("=== MODEL COMPARISON ===")
    display(comparison)

    print("\nSaved:", csv_path)
    print("Saved:", json_path)

# Record research limitations alongside the results
notes = {
    "purpose": "Experimental ASD image-only and questionnaire-only model comparison",
    "test_split_size": 200,
    "decision_threshold": 0.5,
    "important_limitations": [
        "Results are from one held-out split and do not establish generalization.",
        "Questionnaire performance is unusually high and warrants leakage and provenance checks.",
        "Participant-level correspondence between facial images and questionnaire records has not been established.",
        "Do not describe the prototype as clinically validated.",
        "The questionnaire-only model is not a multimodal fusion model."
    ]
}
with open(os.path.join(out, "research_limitations.json"), "w") as f:
    json.dump(notes, f, indent=2)

# Package all experiment artifacts into one recovery ZIP
archive = "/kaggle/working/asd_care_final_backup.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(out):
        for name in files:
            path = os.path.join(root, name)
            z.write(path, os.path.relpath(path, "/kaggle/working"))

print("\nBackup ZIP:", archive)
print("Backup size (MB):", round(os.path.getsize(archive) / (1024 * 1024), 2))
print("Files packaged:", len(zipfile.ZipFile(archive).namelist()))
print("\nIMPORTANT: Download this ZIP to your computer.")




In [ ]:

# Cell: Audit Image + Questionnaire Pairing
import os
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/ajithdari/autism-spectrum-disorder-multi-modal-dataset/Dataset_csv"
FILES = {
    "train": os.path.join(DATA_DIR, "train.csv"),
    "valid": os.path.join(DATA_DIR, "valid.csv"),
    "test":  os.path.join(DATA_DIR, "test.csv"),
}

dfs = {}
for split, path in FILES.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {split} CSV: {path}")
    dfs[split] = pd.read_csv(path)

features = [
    c for c in dfs["train"].columns
    if c not in ["image_path", "Class"]
]

print("=== 1. DATASET STRUCTURE ===")
for split, df in dfs.items():
    print(f"\n{split.upper()}: {df.shape}")
    print("Columns:", df.columns.tolist())
    print("Class counts:")
    print(df["Class"].value_counts(dropna=False).to_dict())
    print("Missing values:", int(df.isna().sum().sum()))
    print("Missing image paths:", int((~df["image_path"].astype(str).map(os.path.exists)).sum()))

print("\n=== 2. IMAGE PATH OVERLAP ACROSS SPLITS ===")
for a, b in [("train", "valid"), ("train", "test"), ("valid", "test")]:
    paths_a = set(dfs[a]["image_path"].astype(str).str.strip())
    paths_b = set(dfs[b]["image_path"].astype(str).str.strip())
    print(f"{a} vs {b}: {len(paths_a & paths_b)} shared paths")

print("\n=== 3. DUPLICATE IMAGE PATHS WITHIN EACH SPLIT ===")
for split, df in dfs.items():
    print(f"{split}: {int(df['image_path'].astype(str).duplicated().sum())}")

print("\n=== 4. QUESTIONNAIRE PATTERN AUDIT ===")
for split, df in dfs.items():
    available = [c for c in features if c in df.columns]
    pattern_counts = df[available].astype(str).value_counts()
    print(
        f"{split}: {len(pattern_counts)} unique questionnaire patterns "
        f"out of {len(df)} rows; repeated-pattern rows = "
        f"{int(df[available].astype(str).duplicated(keep=False).sum())}"
    )

print("\n=== 5. IDENTICAL QUESTIONNAIRE PATTERNS ACROSS SPLITS ===")
pattern_sets = {}
for split, df in dfs.items():
    available = [c for c in features if c in df.columns]
    pattern_sets[split] = set(
        map(tuple, df[available].fillna("<MISSING>").astype(str).to_numpy())
    )

for a, b in [("train", "valid"), ("train", "test"), ("valid", "test")]:
    shared = pattern_sets[a] & pattern_sets[b]
    print(f"{a} vs {b}: {len(shared)} shared questionnaire patterns")

print("\n=== 6. IMAGE / QUESTIONNAIRE / LABEL ROW CHECK ===")
for split, df in dfs.items():
    print(f"\n{split.upper()} first 5 rows:")
    display(df[["image_path"] + features + ["Class"]].head())

print("\nAUDIT COMPLETE.")
print("Important: matching rows in a CSV shows dataset-level association,")
print("but does not independently prove participant identity or clinical pairing.")


# 8. Saved Results and Data-Provenance Review

Read-only audits of existing outputs, questionnaire preprocessing, and data provenance.

In [ ]:
from pathlib import Path

for folder in [
    Path("/kaggle/working"),
    Path("/kaggle/working/asd_care_results"),
    Path("/kaggle/working/asd_care_download_batches"),
]:
    print(f"\n--- {folder} ---")
    if folder.exists():
        for item in sorted(folder.iterdir()):
            if item.is_file():
                print(f"{item.name} | {item.stat().st_size / (1024**2):.2f} MB")
            else:
                print(f"{item.name}/")
    else:
        print("Folder does not exist")

In [ ]:

# Cell 2 — Inspect backup contents
from pathlib import Path

backup = Path("/kaggle/input/datasets/shahbaz18/backup")

print("=== ALL BACKUP FILES ===")
for p in sorted(backup.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(backup)}  |  {p.stat().st_size / 1024:.1f} KB")

print("\n=== RESULT / PREDICTION FILES ===")
keywords = ("result", "predict", "history", "metric", "comparison",
            "test", "label", "scaler", "preprocess", "class")

for p in sorted(backup.rglob("*")):
    if p.is_file() and any(k in p.name.lower() for k in keywords):
        print(p)


In [ ]:

# Cell 3 — Audit saved metrics, predictions, and comparison
from pathlib import Path
import json
import numpy as np

results_dir = Path(
    "/kaggle/input/datasets/shahbaz18/backup/asd_care_results"
)

for name in [
    "mobilenetv2_test_results.json",
    "efficientnetb0_test_results.json",
    "resnet50_test_results.json",
    "questionnaire_mlp_test_results.json",
    "final_model_comparison.json",
    "research_limitations.json",
]:
    path = results_dir / name
    print(f"\n{'=' * 15} {name} {'=' * 15}")
    if path.exists():
        try:
            print(json.dumps(json.loads(path.read_text()), indent=2))
        except Exception as e:
            print("Could not read JSON:", e)
    else:
        print("NOT FOUND")

print("\n========== PREDICTION ARRAYS ==========")
for path in sorted(results_dir.glob("*_test_predictions.npz")):
    print(f"\n{path.name}")
    with np.load(path, allow_pickle=False) as data:
        for key in data.files:
            arr = data[key]
            print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")
            if arr.size <= 12:
                print("   values:", arr.tolist())

print("\nAudit complete. No models were trained or modified.")


In [ ]:

# Cell 4 — Verify saved predictions and recompute metrics
from pathlib import Path
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

results_dir = Path(
    "/kaggle/input/datasets/shahbaz18/backup/asd_care_results"
)

models = [
    "mobilenetv2",
    "efficientnetb0",
    "resnet50",
    "questionnaire_mlp",
]

saved_labels = {}

for model in models:
    path = results_dir / f"{model}_test_predictions.npz"

    with np.load(path, allow_pickle=False) as data:
        y_true = data["y_true"].astype(int)
        y_prob = data["y_prob"].astype(float).ravel()
        y_pred = data["y_pred"].astype(int).ravel()

    saved_labels[model] = y_true

    # Check stored predictions against the 0.5 threshold
    threshold_check = np.array_equal(y_pred, (y_prob >= 0.5).astype(int))

    print(f"\n=== {model.upper()} ===")
    print("Samples:", len(y_true))
    print("Positive labels:", int(y_true.sum()))
    print("Probability range:", (float(y_prob.min()), float(y_prob.max())))
    print("Stored predictions match threshold 0.5:", threshold_check)
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
    print("F1:", round(f1_score(y_true, y_pred, zero_division=0), 4))
    print("ROC-AUC:", round(roc_auc_score(y_true, y_prob), 4))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

print("\n=== TEST LABEL CONSISTENCY ===")
reference = saved_labels[models[0]]

for model in models[1:]:
    same = np.array_equal(reference, saved_labels[model])
    print(f"{model}: same test labels as MobileNetV2 = {same}")

print("\nAudit finished. No model training performed.")


In [ ]:

# Cell 5 — Audit questionnaire preprocessing and data provenance
from pathlib import Path
import json

backup = Path("/kaggle/input/datasets/shahbaz18/backup/asd_care_results")

for filename in [
    "questionnaire_preprocessing.json",
    "questionnaire_scaler.json",
    "questionnaire_mlp_history.json",
]:
    path = backup / filename
    print(f"\n{'=' * 12} {filename} {'=' * 12}")

    if path.exists():
        try:
            obj = json.loads(path.read_text())
            print(json.dumps(obj, indent=2)[:12000])
        except Exception as e:
            print("Read error:", e)
    else:
        print("File not found")

print("\n========== CSV FILES IN ATTACHED DATASETS ==========")
root = Path("/kaggle/input/datasets")
csv_files = list(root.rglob("*.csv"))

if not csv_files:
    print("No CSV files found in attached datasets.")
else:
    for p in csv_files[:100]:
        print(p)

print(f"\nCSV files found: {len(csv_files)}")
print("\nAudit only — no training or file changes.")



In [ ]:

# Cell 6 — Audit questionnaire dataset splits
import pandas as pd
from pathlib import Path

data_dir = Path(
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset/Dataset_csv"
)

files = {
    "train": data_dir / "train.csv",
    "valid": data_dir / "valid.csv",
    "test": data_dir / "test.csv",
}

dfs = {}

for split, path in files.items():
    print(f"\n{'=' * 18} {split.upper()} {'=' * 18}")

    if not path.exists():
        print("FILE NOT FOUND:", path)
        continue

    df = pd.read_csv(path)
    dfs[split] = df

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    # Show distributions only for likely target/label columns
    targets = [
        c for c in df.columns
        if any(k in c.lower() for k in
               ["class", "label", "target", "autism", "asd", "diagnos", "dx_group"])
    ]

    for col in targets:
        print(f"\nPossible label: {col}")
        print(df[col].value_counts(dropna=False).head(15).to_dict())

# Check overlaps using shared columns that look like participant identifiers
if len(dfs) >= 2:
    print("\n========== POSSIBLE ID OVERLAP ==========")

    splits = list(dfs.keys())
    for i in range(len(splits)):
        for j in range(i + 1, len(splits)):
            a_name, b_name = splits[i], splits[j]
            a, b = dfs[a_name], dfs[b_name]

            common = set(a.columns) & set(b.columns)
            id_cols = [
                c for c in common
                if any(k in c.lower() for k in
                       ["participant", "subject", "person_id", "patient_id",
                        "record_id", "sample_id", "nd ar", "guid", "id"])
            ]

            if not id_cols:
                print(f"{a_name} vs {b_name}: no obvious shared ID column")
                continue

            for col in id_cols:
                av = set(a[col].dropna().astype(str))
                bv = set(b[col].dropna().astype(str))
                overlap = av & bv
                print(
                    f"{a_name} vs {b_name} | {col}: "
                    f"{len(overlap)} overlapping values "
                    f"(A unique={len(av)}, B unique={len(bv)})"
                )

print("\nAudit complete — no data or models modified.")


In [ ]:

# Cell 7 — Audit questionnaire labels and image-path overlap
# AUDIT ONLY: does not train models or modify saved files.

import pandas as pd
from pathlib import Path
from collections import Counter

data_dir = Path(
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset/Dataset_csv"
)

dfs = {
    split: pd.read_csv(data_dir / f"{split}.csv")
    for split in ["train", "valid", "test"]
}

# 1. Check whether Class is directly determined by A1–A10
for split, df in dfs.items():
    print(f"\n===== {split.upper()}: LABEL RELATIONSHIP =====")

    answers = [f"A{i}" for i in range(1, 11)]
    missing = [c for c in answers + ["Class", "image_path"] if c not in df.columns]

    if missing:
        print("Missing columns:", missing)
        continue

    print("Class distribution:")
    print(df["Class"].value_counts(dropna=False).to_dict())

    df["_answer_sum"] = df[answers].apply(
        pd.to_numeric, errors="coerce"
    ).sum(axis=1, min_count=10)

    print("\nAnswer-sum summary by Class:")
    print(df.groupby("Class")["_answer_sum"].agg(
        ["count", "min", "median", "max", "mean"]
    ).to_string())

    print("\nClass vs answer sum (top combinations):")
    print(pd.crosstab(df["_answer_sum"], df["Class"]).head(20).to_string())

    print("\nA1–A10 missing-value counts:")
    print(df[answers].isna().sum().to_dict())

# 2. Check exact image path overlap between splits
print("\n===== EXACT IMAGE-PATH OVERLAP =====")

path_sets = {}
for split, df in dfs.items():
    paths = df["image_path"].dropna().astype(str).str.strip()
    path_sets[split] = set(paths)
    print(
        f"{split}: rows={len(df)}, unique_paths={paths.nunique()}, "
        f"missing_paths={df['image_path'].isna().sum()}"
    )

for a, b in [("train", "valid"), ("train", "test"), ("valid", "test")]:
    common = path_sets[a] & path_sets[b]
    print(f"{a} vs {b}: {len(common)} identical image_path values")
    if common:
        print("Examples:", list(common)[:5])

# 3. Check whether answer patterns repeat with conflicting labels
print("\n===== REPEATED ANSWER PATTERNS =====")
all_df = pd.concat(
    [df.assign(_split=split) for split, df in dfs.items()],
    ignore_index=True
)

answers = [f"A{i}" for i in range(1, 11)]
all_df["_answer_pattern"] = (
    all_df[answers].astype(str).agg("|".join, axis=1)
)

pattern_label_counts = all_df.groupby("_answer_pattern")["Class"].nunique()
conflicting = pattern_label_counts[pattern_label_counts > 1]

print("Unique answer patterns:", all_df["_answer_pattern"].nunique())
print("Patterns associated with multiple Class labels:", len(conflicting))

print("\nAudit complete. No models or source CSVs modified.")


In [ ]:

# Cell 8 — Verify saved models and final results
# No training. No files modified.

from pathlib import Path
import json
import pandas as pd

root = Path("/kaggle/input/datasets/shahbaz18/backup/asd_care_results")

assert root.exists(), (
    f"Backup not found: {root}\n"
    "Make sure the Kaggle dataset 'backup' is attached."
)

comparison = root / "final_model_comparison.csv"

if comparison.exists():
    print("===== SAVED MODEL COMPARISON =====")
    print(pd.read_csv(comparison).to_string(index=False))
else:
    print("Comparison CSV not found; listing individual results.")

    for p in sorted(root.glob("*test_results.json")):
        print(f"\n===== {p.name} =====")
        with open(p, "r") as f:
            print(json.dumps(json.load(f), indent=2))

print("\n===== REQUIRED ARTIFACTS =====")
for pattern in [
    "*best.keras",
    "*test_predictions.npz",
    "*history.json",
    "questionnaire_preprocessing.json",
    "research_limitations.json"
]:
    found = sorted(root.glob(pattern))
    print(f"\n{pattern}: {len(found)} file(s)")
    for p in found:
        print(f"  {p.name} ({p.stat().st_size / 1024 / 1024:.2f} MB)")


In [ ]:

# Cell 9 — Save final results and limitations in Kaggle
# Uses the existing comparison CSV; no model training.

from pathlib import Path
import json
import pandas as pd

root = Path("/kaggle/input/datasets/shahbaz18/backup/asd_care_results")
out = Path("/kaggle/working/asd_care_final_report")
out.mkdir(parents=True, exist_ok=True)

comparison_path = root / "final_model_comparison.csv"
assert comparison_path.exists(), f"Missing: {comparison_path}"

comparison = pd.read_csv(comparison_path)
comparison.to_csv(out / "final_model_comparison.csv", index=False)
comparison.to_json(
    out / "final_model_comparison.json",
    orient="records",
    indent=2
)

limitations = {
    "project_type": "Research prototype",
    "clinical_validation": False,
    "image_models": ["MobileNetV2", "EfficientNetB0", "ResNet50"],
    "questionnaire_model": "Separate questionnaire-only baseline",
    "questionnaire_caveat": (
        "Class labels are strongly associated with the sum of A1-A10. "
        "The high questionnaire accuracy may reflect label construction."
    ),
    "fusion_caveat": (
        "Participant-level image-questionnaire matching is unverified. "
        "Do not report fused multimodal performance without verified pairs."
    ),
    "generalization_caveat": (
        "Results come from one held-out test split and do not establish "
        "external generalization."
    )
}

with open(out / "research_limitations.json", "w", encoding="utf-8") as f:
    json.dump(limitations, f, indent=2)

print("Saved final comparison and limitations to:", out)
print("\nComparison:")
print(comparison.to_string(index=False))
print("\nCreated files:")
for p in sorted(out.iterdir()):
    print(p.name, f"{p.stat().st_size:,} bytes")



In [ ]:

# Cell 10 — Create one final ZIP package (no training)

from pathlib import Path
import shutil
import json

source = Path("/kaggle/input/datasets/shahbaz18/backup/asd_care_results")
report = Path("/kaggle/working/asd_care_final_report")
package = Path("/kaggle/working/asd_care_final_package")

package.mkdir(parents=True, exist_ok=True)

# Copy final comparison and research limitations
for name in [
    "final_model_comparison.csv",
    "final_model_comparison.json",
    "research_limitations.json"
]:
    src = report / name
    if src.exists():
        shutil.copy2(src, package / name)

# Include saved models, prediction files, histories and result JSONs
patterns = [
    "*_best.keras",
    "*_test_predictions.npz",
    "*_test_results.json",
    "*history*.json",
    "*training*.log",
    "questionnaire_preprocessing.json",
    "questionnaire_scaler.json",
]

copied = set()
for pattern in patterns:
    for src in source.glob(pattern):
        if src.is_file() and src.name not in copied:
            shutil.copy2(src, package / src.name)
            copied.add(src.name)

# Include any existing training logs with other extensions
for src in source.glob("*"):
    if src.is_file() and (
        "training" in src.name.lower()
        or "history" in src.name.lower()
    ):
        if src.name not in copied:
            shutil.copy2(src, package / src.name)
            copied.add(src.name)

# Write a README with interpretation cautions
readme = """
ASD CARE — FINAL RESEARCH ARTIFACTS

Included:
- Final model comparison
- Research limitations
- Saved model checkpoints
- Test prediction arrays and test result JSON files
- Available training histories/logs
- Questionnaire preprocessing/scaler files, if present

IMAGE-ONLY TEST RESULTS:
ResNet50:       Accuracy 0.890, ROC-AUC 0.9509
MobileNetV2:    Accuracy 0.850, ROC-AUC 0.9307
EfficientNetB0: Accuracy 0.815, ROC-AUC 0.9347

QUESTIONNAIRE BASELINE:
Accuracy 0.985, ROC-AUC 0.9998.
Important: questionnaire labels are strongly associated with A1-A10
answer sums. This result may reflect label construction and should not
be presented as independent proof of autism detection.

IMPORTANT LIMITATIONS:
- Research prototype only; not clinically validated.
- Participant-level matching of images and questionnaire records
  has not been verified.
- Do not claim multimodal fusion results without verified paired data.
- One held-out test split does not establish external generalization.

Use the saved predictions for further analysis; do not retrain unless needed.
"""
(package / "README.txt").write_text(readme.strip() + "\n", encoding="utf-8")

# Create downloadable archive
zip_base = Path("/kaggle/working/asd_care_final_package")
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=package)

print("PACKAGE CREATED:", zip_path)
print("Package size (MB):", round(Path(zip_path).stat().st_size / (1024**2), 2))
print("\nIncluded files:")
for p in sorted(package.iterdir()):
    print(f"{p.name} — {p.stat().st_size:,} bytes")


# 9. Pairing, Leakage, and Split Integrity

Document pseudo-pairing limitations and check for label or split leakage before interpreting fusion results.

In [ ]:

# Cell 11 — Verify whether questionnaire records link to real images

from pathlib import Path
import pandas as pd

data_dir = Path(
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset/Dataset_csv"
)

for split in ["train", "valid", "test"]:
    csv_path = data_dir / f"{split}.csv"
    df = pd.read_csv(csv_path)

    print(f"\n===== {split.upper()} =====")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    if "image_path" not in df.columns:
        print("STOP: No image_path column.")
        continue

    paths = df["image_path"].dropna().astype(str)
    print("Example image paths:")
    for p in paths.head(5):
        print(" ", p)

    exists_count = 0
    for p in paths:
        candidate = Path(p)
        if candidate.is_file():
            exists_count += 1
        elif (data_dir / p).is_file():
            exists_count += 1

    print(f"Image files found: {exists_count}/{len(paths)}")
    print("Unique image paths:", paths.nunique())
    print("Label counts:")
    print(df["Class"].value_counts(dropna=False).to_string())

print("\nAudit complete. No models were trained or changed.")


In [ ]:

# Cell 12 — Final pairing and label-leakage audit (no training)

from pathlib import Path
import pandas as pd

data_dir = Path(
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset/Dataset_csv"
)

answer_cols = [f"A{i}" for i in range(1, 11)]

for split in ["train", "valid", "test"]:
    df = pd.read_csv(data_dir / f"{split}.csv")
    df["Class"] = df["Class"].astype(str).str.strip().str.upper()
    df["answer_sum"] = df[answer_cols].apply(
        pd.to_numeric, errors="coerce"
    ).sum(axis=1)

    # Infer image label from the parent folder name.
    folder = df["image_path"].astype(str).apply(
        lambda p: Path(p).parent.name.lower()
    )

    def infer_image_label(name):
        if "typical" in name or name in {"td", "control", "controls"}:
            return "NO"
        if "autistic" in name or name == "asd":
            return "YES"
        return "UNKNOWN"

    df["image_label"] = folder.apply(infer_image_label)

    print(f"\n===== {split.upper()} =====")
    print("Image-folder labels:")
    print(df["image_label"].value_counts().to_string())

    known = df[df["image_label"] != "UNKNOWN"]
    if len(known):
        print("\nImage-folder label vs CSV Class:")
        print(pd.crosstab(known["image_label"], known["Class"]).to_string())
        print(
            "Agreement:",
            f"{(known['image_label'] == known['Class']).mean():.1%}"
        )

    print("\nQuestionnaire answer-sum ranges by Class:")
    print(
        df.groupby("Class")["answer_sum"]
          .agg(["count", "min", "median", "max", "mean"])
          .to_string()
    )

    print("\nClass by answer-sum range:")
    bins = [-1, 5, 6, 10, 100]
    labels = ["0–5", "6", "7–10", "11+"]
    df["sum_range"] = pd.cut(
        df["answer_sum"], bins=bins, labels=labels
    )
    print(pd.crosstab(df["sum_range"], df["Class"]).to_string())

print(
    "\nIMPORTANT: Matching file paths and matching labels do not prove "
    "that the questionnaire respondent is the person pictured."
)


In [ ]:

# Cell 13 — Inspect image folders and dataset documentation (no training)

from pathlib import Path

roots = [
    Path("/kaggle/input/datasets/prayashdas/autistic-children-facial-image-dataset"),
    Path("/kaggle/input/datasets/ajithdari/autism-spectrum-disorder-multi-modal-dataset"),
]

for root in roots:
    print(f"\n===== ROOT: {root} =====")
    print("Exists:", root.exists())

    if not root.exists():
        continue

    # Show directory structure to depth 3
    for p in sorted(root.rglob("*")):
        if p.is_dir():
            try:
                depth = len(p.relative_to(root).parts)
            except ValueError:
                depth = 99
            if depth <= 3:
                print("DIR ", p.relative_to(root))

    # Show likely documentation files
    docs = [
        p for p in root.rglob("*")
        if p.is_file()
        and p.suffix.lower() in {".txt", ".md", ".pdf", ".json"}
        and p.stat().st_size < 2_000_000
    ]

    print("\nPossible documentation files:")
    for p in docs[:30]:
        print("FILE", p.relative_to(root))


In [ ]:

# Cell 14 — Verify CSV rows against actual Dataset_img class folders
# Read-only audit; no training or file changes.

from pathlib import Path
import pandas as pd

base = Path(
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset"
)
csv_dir = base / "Dataset_csv"
img_root = base / "Dataset_img"

for split in ["train", "valid", "test"]:
    df = pd.read_csv(csv_dir / f"{split}.csv")
    df["Class"] = df["Class"].astype(str).str.strip().str.upper()

    checked = 0
    found = 0
    matching = 0
    missing = []
    mismatches = []

    for _, row in df.iterrows():
        raw = str(row["image_path"])
        filename = Path(raw).name
        label = row["Class"]

        # Resolve the image inside this dataset's own image folders.
        candidates = [
            img_root / split / "autistic" / filename,
            img_root / split / "non_autistic" / filename,
        ]
        existing = [p for p in candidates if p.is_file()]

        checked += 1
        if len(existing) == 1:
            found += 1
            actual = existing[0].parent.name
            expected = "autistic" if label == "YES" else "non_autistic"

            if actual == expected:
                matching += 1
            elif len(mismatches) < 10:
                mismatches.append(
                    (filename, label, actual)
                )
        else:
            if len(missing) < 10:
                missing.append((filename, len(existing)))

    print(f"\n===== {split.upper()} =====")
    print("CSV rows checked:", checked)
    print("Unique image files found in Dataset_img:", found)
    print("Class-folder agreement:", f"{matching}/{found}" if found else "N/A")
    print("Example missing/ambiguous files:", missing)
    print("Example class mismatches:", mismatches)

print("\nThis verifies filename and class-folder consistency only.")
print("It cannot prove participant identity or questionnaire provenance.")


In [ ]:

# Cell 15 — Check actual image filenames in both classes

from pathlib import Path
import pandas as pd

base = Path(
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset"
)
img_root = base / "Dataset_img"
csv_dir = base / "Dataset_csv"

for split in ["train", "valid", "test"]:
    print(f"\n===== {split.upper()} =====")

    df = pd.read_csv(csv_dir / f"{split}.csv")

    for cls in ["autistic", "non_autistic"]:
        folder = img_root / split / cls
        files = sorted(
            p.name for p in folder.iterdir()
            if p.is_file()
        ) if folder.exists() else []

        print(f"{cls}: {len(files)} files")
        print("Examples:", files[:8])

    autistic = {
        p.name for p in (img_root / split / "autistic").iterdir()
        if p.is_file()
    }
    non_autistic = {
        p.name for p in (img_root / split / "non_autistic").iterdir()
        if p.is_file()
    }

    print("Filenames present in BOTH classes:", len(autistic & non_autistic))
    print("CSV rows:", len(df))
    print("CSV image filename examples:",
          [Path(str(p)).name for p in df["image_path"].head(8)])

print("\nNo files modified; no training performed.")


# 10. Multimodal Prototype Preparation and Late Fusion

Prepare branch inputs/predictions, then evaluate the 50:50 late-fusion experiment without retraining where possible.

In [ ]:

# Cell 16 — Build an auditable pseudo-paired multimodal manifest
# No model training. Original files and checkpoints remain unchanged.
# IMPORTANT: Pairing is by class, NOT verified participant identity.

from pathlib import Path
import pandas as pd
import numpy as np
import json

csv_root = Path(
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset/Dataset_csv"
)
image_root = Path(
    "/kaggle/input/datasets/prayashdas/"
    "autistic-children-facial-image-dataset"
)
out = Path("/kaggle/working/asd_pseudo_multimodal")
out.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(2026)
all_parts = []
audit = {
    "pairing_type": "synthetic class-stratified pseudo-pairing",
    "participant_identity_verified": False,
    "clinical_multimodal_validity_established": False,
    "seed": 2026,
    "splits": {}
}

for split in ["train", "valid", "test"]:
    df = pd.read_csv(csv_root / f"{split}.csv").copy()
    df["Class"] = df["Class"].astype(str).str.strip().str.upper()

    if not set(df["Class"].unique()).issubset({"YES", "NO"}):
        raise ValueError(f"Unexpected Class values in {split}")

    df["image_filename"] = df["image_path"].astype(str).map(
        lambda p: Path(p).name
    )

    # Use the correct class-specific folder.
    # This selects an image by its class label; it does NOT establish identity.
    df["image_class_folder"] = df["Class"].map({
        "YES": "autistic",
        "NO": "non_autistic"
    })
    df["resolved_image_path"] = df.apply(
        lambda r: str(
            image_root / split / r["image_class_folder"] /
            r["image_filename"]
        ),
        axis=1
    )

    missing = [
        p for p in df["resolved_image_path"]
        if not Path(p).is_file()
    ]
    if missing:
        raise FileNotFoundError(
            f"{split}: {len(missing)} class-specific images missing. "
            f"Examples: {missing[:5]}"
        )

    # Unique row ID; not a real participant ID.
    df["sample_id"] = [
        f"{split.upper()}_{i:05d}" for i in range(1, len(df) + 1)
    ]

    # Explicitly label pairing status for downstream reporting.
    df["pairing_status"] = "synthetic_class_stratified_unverified"
    df["participant_id"] = pd.NA

    # Keep all source questionnaire features and provenance columns.
    all_parts.append(df)

    audit["splits"][split] = {
        "rows": int(len(df)),
        "class_counts": {
            str(k): int(v) for k, v in df["Class"].value_counts().items()
        },
        "missing_resolved_images": 0,
        "unique_sample_ids": int(df["sample_id"].nunique()),
        "participant_ids_verified": False
    }

manifest = pd.concat(all_parts, ignore_index=True)

assert manifest["sample_id"].is_unique
assert manifest["resolved_image_path"].map(
    lambda p: Path(p).is_file()
).all()

manifest_path = out / "pseudo_multimodal_manifest.csv"
manifest.to_csv(manifest_path, index=False)

with open(out / "pairing_audit.json", "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2)

print("Manifest saved:", manifest_path)
print("Audit saved:", out / "pairing_audit.json")
print("Total rows:", len(manifest))
print("Unique sample IDs:", manifest["sample_id"].nunique())
print("All resolved images exist: YES")
print("\nPairing status:")
print(manifest["pairing_status"].value_counts().to_string())
print("\nClass counts by split:")
print(pd.crosstab(manifest["image_class_folder"], manifest["Class"]))
print(
    "\nWARNING: IDs identify dataset rows only. They do not identify people. "
    "This manifest is suitable for a synthetic pairing experiment only."
)


In [ ]:

# Cell 17 — Prepare multimodal prototype inputs
# Existing models remain untouched.

from pathlib import Path
import pandas as pd
import numpy as np
import tensorflow as tf

CSV_ROOT = Path(
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset/Dataset_csv"
)
IMAGE_ROOT = Path(
    "/kaggle/input/datasets/prayashdas/"
    "autistic-children-facial-image-dataset"
)
MODEL_ROOT = Path(
    "/kaggle/input/datasets/shahbaz18/backup/asd_care_results"
)
WORK = Path("/kaggle/working/asd_multimodal_prototype")
WORK.mkdir(parents=True, exist_ok=True)

image_model_path = MODEL_ROOT / "resnet50_best.keras"
questionnaire_model_path = MODEL_ROOT / "questionnaire_mlp_best.keras"

for p in [image_model_path, questionnaire_model_path]:
    assert p.exists(), f"Missing saved model: {p}"
    print(f"Found {p.name}: {p.stat().st_size / 1e6:.2f} MB")

# Load saved models for inference only.
image_model = tf.keras.models.load_model(image_model_path, compile=False)
questionnaire_model = tf.keras.models.load_model(
    questionnaire_model_path, compile=False
)

print("\nImage model input/output:", image_model.input_shape, image_model.output_shape)
print("Questionnaire model input/output:",
      questionnaire_model.input_shape, questionnaire_model.output_shape)

feature_cols = [
    "A1", "A2", "A3", "A4", "A5",
    "A6", "A7", "A8", "A9", "A10",
    "Age", "Sex", "Jauundice", "Family_ASD"
]

prepared = {}

for split in ["train", "valid", "test"]:
    df = pd.read_csv(CSV_ROOT / f"{split}.csv")
    df["Class"] = df["Class"].astype(str).str.strip().str.upper()
    df["label"] = df["Class"].map({"NO": 0, "YES": 1})

    if df["label"].isna().any():
        raise ValueError(f"Unexpected labels in {split}")

    # Resolve each image by the row's class label.
    df["image_path_resolved"] = df.apply(
        lambda r: str(
            IMAGE_ROOT / split /
            ("autistic" if r["label"] == 1 else "non_autistic") /
            Path(str(r["image_path"])).name
        ),
        axis=1
    )

    missing = [
        p for p in df["image_path_resolved"]
        if not Path(p).is_file()
    ]
    if missing:
        raise FileNotFoundError(
            f"{split}: missing images, examples: {missing[:5]}"
        )

    # Numeric, finite questionnaire features.
    Xq = df[feature_cols].copy()
    for c in feature_cols:
        Xq[c] = pd.to_numeric(Xq[c], errors="coerce")
        Xq[c] = Xq[c].fillna(Xq[c].median())
    Xq = Xq.astype("float32")

    prepared[split] = {
        "df": df,
        "image_paths": df["image_path_resolved"].tolist(),
        "Xq": Xq.to_numpy(),
        "y": df["label"].to_numpy(dtype="int32")
    }

    print(
        f"{split}: {len(df)} rows | "
        f"labels={dict(df['label'].value_counts().sort_index())} | "
        f"all images exist"
    )

print("\nPreparation complete. No training performed yet.")
print("Prototype outputs will be saved under:", WORK)
print("Note: this is class-based synthetic pairing, not verified participant matching.")


In [ ]:

# Cell 18A — Diagnose questionnaire input mismatch (no training)


import json
from pathlib import Path

import pandas as pd

MODEL_ROOT = Path(
    "/kaggle/input/datasets/shahbaz18/backup/asd_care_results"
)
CSV_ROOT = Path(
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset/Dataset_csv"
)

for filename in [
    "questionnaire_preprocessing.json",
    "questionnaire_scaler.json",
]:
    path = MODEL_ROOT / filename
    print(f"\n===== {filename} =====")
    if path.exists():
        print(path.read_text(encoding="utf-8")[:12000])
    else:
        print("Not found")

print("\n===== CSV FEATURES =====")
df = pd.read_csv(CSV_ROOT / "train.csv")
print("Columns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes.to_string())

print("\n===== SAVED MLP INPUT =====")
print("Expected feature count:", questionnaire_model.input_shape[-1])

print("\nDiagnostic complete. No models or files changed.")


In [ ]:

# Cell 18B — Fix questionnaire preprocessing and verify predictions
# Safe: does not train or overwrite any saved model.

import json
import numpy as np
import pandas as pd
from pathlib import Path

MODEL_ROOT = Path("/kaggle/input/datasets/shahbaz18/backup/asd_care_results")
with open(MODEL_ROOT / "questionnaire_preprocessing.json", "r") as f:
    prep = json.load(f)

numeric_cols = prep["numeric_features"]
categorical_cols = prep["categorical_features"]
category_levels = prep["category_levels"]

def encode_questionnaire(df):
    df = df.copy()
    parts = []

    # Numeric columns: impute using saved medians, then saved scaling.
    numeric = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
    for col in numeric_cols:
        numeric[col] = numeric[col].fillna(prep["numeric_medians"][col])

    means = np.asarray(prep["scaler_mean"], dtype=np.float32)
    scales = np.asarray(prep["scaler_scale"], dtype=np.float32)
    numeric_array = (numeric.to_numpy(dtype=np.float32) - means) / scales
    parts.append(numeric_array)

    # Categorical columns: preserve saved category order and all levels.
    for col in categorical_cols:
        values = df[col].astype("string").str.strip().str.lower()
        values = values.fillna("unknown")
        levels = category_levels[col]
        encoded = np.column_stack([
            (values.to_numpy() == level).astype(np.float32)
            for level in levels
        ])
        parts.append(encoded)

    X = np.concatenate(parts, axis=1).astype(np.float32)
    expected = int(questionnaire_model.input_shape[-1])
    assert X.shape[1] == expected, (
        f"Feature mismatch: created {X.shape[1]}, expected {expected}"
    )
    return X

for split, data in prepared.items():
    data["Xq"] = encode_questionnaire(data["df"])
    print(f"{split}: questionnaire features = {data['Xq'].shape}")

# Verify one split first; no need to rerun image predictions.
test_data = prepared["test"]
p_q_test = questionnaire_model.predict(
    test_data["Xq"], batch_size=32, verbose=0
).reshape(-1)

print("\nQuestionnaire inference successful.")
print("Test predictions:", p_q_test.shape)
print("Prediction range:", float(p_q_test.min()), "to", float(p_q_test.max()))
print("First 10 probabilities:", np.round(p_q_test[:10], 4))


In [ ]:

# Cell 19 — Prepare cached image and questionnaire predictions

import numpy as np
from pathlib import Path

WORK = Path("/kaggle/working/asd_multimodal_prototype")
WORK.mkdir(parents=True, exist_ok=True)

for split, data in prepared.items():
    cache = WORK / f"{split}_branch_predictions.npz"

    if cache.exists():
        with np.load(cache) as saved:
            print(f"\n{split}: existing cache found")
            print("Keys:", saved.files)
            for key in saved.files:
                print(f"  {key}: {saved[key].shape}")
    else:
        print(f"\n{split}: no prediction cache yet")

print("\nPrepared splits:", list(prepared.keys()))
print("Image model:", image_model.input_shape, "->", image_model.output_shape)
print("Questionnaire model:", questionnaire_model.input_shape,
      "->", questionnaire_model.output_shape)


In [ ]:

# Cell 20 — Generate and save branch predictions (no training)

import json
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.utils import load_img, img_to_array

WORK = Path("/kaggle/working/asd_multimodal_prototype")
WORK.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
IMG_SIZE = (224, 224)

def predict_images(paths):
    results = []
    for start in range(0, len(paths), BATCH_SIZE):
        batch_paths = paths[start:start + BATCH_SIZE]
        batch = np.stack([
            img_to_array(load_img(p, target_size=IMG_SIZE))
            for p in batch_paths
        ]).astype(np.float32)

        batch = preprocess_input(batch)
        pred = image_model.predict(batch, verbose=0).reshape(-1)
        results.extend(pred.tolist())

    return np.asarray(results, dtype=np.float32)

for split, data in prepared.items():
    cache = WORK / f"{split}_branch_predictions.npz"

    # Reuse a complete cache if present.
    if cache.exists():
        with np.load(cache) as z:
            if all(k in z.files for k in ["p_img", "p_q", "y"]):
                print(f"{split}: complete cache already exists; skipping.")
                continue

    print(f"\nProcessing {split} ({len(data['y'])} samples)...")

    p_img = predict_images(data["image_paths"])
    p_q = questionnaire_model.predict(
        data["Xq"], batch_size=BATCH_SIZE, verbose=0
    ).reshape(-1)

    y = np.asarray(data["y"]).reshape(-1).astype(np.int32)

    assert len(p_img) == len(p_q) == len(y)
    np.savez_compressed(
        cache,
        p_img=p_img,
        p_q=p_q,
        y=y
    )

    print("Saved:", cache)
    print("Image probabilities:", p_img.shape,
          "range:", (float(p_img.min()), float(p_img.max())))
    print("Questionnaire probabilities:", p_q.shape,
          "range:", (float(p_q.min()), float(p_q.max())))

print("\nAll split predictions processed.")
print("Cache directory:", WORK)


In [ ]:

# Cell 21 — Test-set ablation comparison (no training)

import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

WORK = Path("/kaggle/working/asd_multimodal_prototype")
RESULTS = []

def evaluate(name, y_true, probabilities):
    probabilities = np.asarray(probabilities).reshape(-1)
    predictions = (probabilities >= 0.5).astype(int)
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "confusion_matrix": confusion_matrix(
            y_true, predictions
        ).tolist()
    }

with np.load(WORK / "test_branch_predictions.npz") as z:
    y_test = z["y"].astype(int)
    p_img = z["p_img"]
    p_q = z["p_q"]

assert len(y_test) == len(p_img) == len(p_q) == 200
assert set(np.unique(y_test)).issubset({0, 1})

# Simple late fusion: equal weights, fixed before test evaluation.
p_fusion = 0.5 * p_img + 0.5 * p_q

RESULTS.append(evaluate("ResNet50 image-only", y_test, p_img))
RESULTS.append(evaluate("Questionnaire MLP", y_test, p_q))
RESULTS.append(evaluate("Late fusion (50:50)", y_test, p_fusion))

comparison = pd.DataFrame([
    {k: v for k, v in row.items() if k != "confusion_matrix"}
    for row in RESULTS
])

print("TEST SET COMPARISON")
display(comparison.round(4))

for row in RESULTS:
    print(f"\n{row['model']}")
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(np.array(row["confusion_matrix"]))

comparison.to_csv(WORK / "test_ablation_comparison.csv", index=False)

with open(WORK / "test_ablation_results.json", "w") as f:
    json.dump(RESULTS, f, indent=2)

print("\nSaved comparison and detailed metrics to:", WORK)
print("IMPORTANT: image-questionnaire pairing is synthetic and unverified.")


In [ ]:
# Final artifact check (no training)

from pathlib import Path

BACKUP = Path(
    "/kaggle/input/datasets/shahbaz18/backup/asd_care_results"
)
FUSION = Path(
    "/kaggle/working/asd_multimodal_prototype"
)

required = {
    "MobileNetV2 model": BACKUP / "mobilenetv2_best.keras",
    "EfficientNetB0 model": BACKUP / "efficientnetb0_best.keras",
    "ResNet50 model": BACKUP / "resnet50_best.keras",
    "Questionnaire model": BACKUP / "questionnaire_mlp_best.keras",
    "Fusion test predictions": FUSION / "test_branch_predictions.npz",
    "Fusion comparison CSV": FUSION / "test_ablation_comparison.csv",
    "Fusion detailed results": FUSION / "test_ablation_results.json",
}

for name, path in required.items():
    print(f"{'OK' if path.exists() else 'MISSING'} | {name}")

print("\nCheck complete. No models trained.")

# 11. Final Report, Diagnostics, and Error Analysis

Generate the final report and inspect branch contributions, errors, and questionnaire leakage.

In [ ]:

# Cell 24 — Final report and downloadable package (NO TRAINING)

from pathlib import Path
import json
import shutil
import zipfile
import pandas as pd

BACKUP = Path("/kaggle/input/datasets/shahbaz18/backup/asd_care_results")
FUSION = Path("/kaggle/working/asd_multimodal_prototype")
OUT = Path("/kaggle/working/asd_fyp_final_results")
ZIP_PATH = Path("/kaggle/working/asd_fyp_final_results.zip")

OUT.mkdir(parents=True, exist_ok=True)
(OUT / "models").mkdir(exist_ok=True)
(OUT / "fusion_results").mkdir(exist_ok=True)

# 1. Copy model files. Questionnaire model remains an internal
# dependency for the fusion experiment, not a separate result section.
model_files = [
    "mobilenetv2_best.keras",
    "efficientnetb0_best.keras",
    "resnet50_best.keras",
    "questionnaire_mlp_best.keras",
]

for name in model_files:
    src = BACKUP / name
    if src.exists():
        shutil.copy2(src, OUT / "models" / name)
        print("Copied model:", name)
    else:
        print("MISSING MODEL:", name)

# 2. Copy fusion artifacts for reproducibility.
# Original experiment artifacts are preserved.
for name in [
    "train_branch_predictions.npz",
    "valid_branch_predictions.npz",
    "test_branch_predictions.npz",
    "test_ablation_comparison.csv",
    "test_ablation_results.json",
]:
    src = FUSION / name
    if src.exists():
        shutil.copy2(src, OUT / "fusion_results" / name)
        print("Copied fusion artifact:", name)
    else:
        print("Not found (skipped):", name)

# 3. Copy available histories, predictions, comparisons and notes.
# Avoid copying any of these over the final report files.
for pattern in [
    "*comparison*.csv",
    "*comparison*.json",
    "*history*.json",
    "*predictions*.npz",
    "research_limitations.json",
    "questionnaire_preprocessing.json",
    "questionnaire_scaler.json",
]:
    for src in BACKUP.glob(pattern):
        if src.is_file():
            dest = OUT / src.name
            if not dest.exists():
                shutil.copy2(src, dest)

# 4. Three image-only result rows.
# ResNet50 precision and F1 are corrected using its confusion matrix:
# [[88, 12], [9, 91]]
image_results = [
    {
        "section": "1. MobileNetV2",
        "accuracy": 0.8500,
        "precision": 0.8571,
        "recall": 0.8400,
        "f1_score": 0.8485,
        "roc_auc": 0.9307,
        "evaluation": "Image-only",
    },
    {
        "section": "2. EfficientNetB0",
        "accuracy": 0.8150,
        "precision": 0.8889,
        "recall": 0.7200,
        "f1_score": 0.7956,
        "roc_auc": 0.9347,
        "evaluation": "Image-only",
    },
    {
        "section": "3. ResNet50",
        "accuracy": 0.8950,
        "precision": 91 / (91 + 12),
        "recall": 91 / (91 + 9),
        "f1_score": 2 * (91 / 103) * (91 / 100)
                    / ((91 / 103) + (91 / 100)),
        "roc_auc": 0.9509,
        "evaluation": "Image-only",
    },
]

# 5. Read saved fusion results, but select ONLY the late-fusion row.
fusion_json = FUSION / "test_ablation_results.json"

if not fusion_json.exists():
    raise FileNotFoundError(
        f"Missing saved fusion results: {fusion_json}"
    )

fusion_data = json.loads(fusion_json.read_text(encoding="utf-8"))

# Support either a list of rows or a dictionary containing a result list.
if isinstance(fusion_data, dict):
    for key in ["results", "models", "comparison", "ablation_results"]:
        if isinstance(fusion_data.get(key), list):
            fusion_data = fusion_data[key]
            break
    else:
        fusion_data = [fusion_data]

if not isinstance(fusion_data, list):
    raise ValueError("Unexpected format in test_ablation_results.json")

late_fusion_candidates = []
for row in fusion_data:
    if not isinstance(row, dict):
        continue

    label = str(
        row.get("model", row.get("section", row.get("name", "")))
    ).lower().replace("_", " ").replace("-", " ")

    # Exclude standalone questionnaire and image-only rows.
    if "fusion" in label and "questionnaire mlp" not in label:
        late_fusion_candidates.append(row)

if not late_fusion_candidates:
    print("Available saved fusion entries:")
    print(fusion_data)
    raise ValueError(
        "Could not identify a late-fusion result. "
        "Inspect the printed entries and adjust the selection "
        "to match the exact saved model name. No metrics were invented."
    )

# Choose the explicit late-fusion entry when several fusion variants exist.
preferred = [
    row for row in late_fusion_candidates
    if "late fusion" in str(
        row.get("model", row.get("section", row.get("name", "")))
    ).lower().replace("_", " ").replace("-", " ")
]

fusion_row = (preferred or late_fusion_candidates)[0]

final_fusion_result = {
    "section": "4. Late Fusion (50:50)",
    "accuracy": fusion_row.get("accuracy"),
    "precision": fusion_row.get("precision"),
    "recall": fusion_row.get("recall"),
    "f1_score": fusion_row.get("f1_score", fusion_row.get("f1")),
    "roc_auc": fusion_row.get("roc_auc"),
    "evaluation": "Image-questionnaire late fusion",
}

all_results = image_results + [final_fusion_result]

# Enforce exactly four thesis-facing result rows.
final_df = pd.DataFrame(all_results)
assert len(final_df) == 4
assert not final_df["section"].str.contains(
    "questionnaire mlp", case=False, na=False
).any()

# 6. Save final four-row CSV and JSON.
final_df.to_csv(OUT / "final_four_section_results.csv", index=False)

with open(OUT / "final_four_section_results.json", "w",
          encoding="utf-8") as f:
    json.dump(all_results, f, indent=2)

# 7. Write README clarifying what is and is not in the main comparison.
readme = """ASD FYP — Final Results Package

Thesis-facing result sections:
1. MobileNetV2 — image-only
2. EfficientNetB0 — image-only
3. ResNet50 — image-only
4. Late Fusion (50:50) — image + questionnaire

The standalone Questionnaire MLP result is excluded from the
final four-section comparison. The questionnaire model file and
original fusion experiment artifacts remain in the package for
implementation and reproducibility.

Methodological limitation:
Participant identity for image-questionnaire pairs was not verified.
The fusion experiment must therefore be described as exploratory,
not as verified participant-level multimodal evaluation.
Potential questionnaire-label association/leakage should be discussed.

These results do not establish clinical validity.
This packaging cell performs no model training.
"""

(OUT / "README.txt").write_text(readme, encoding="utf-8")

# 8. Rebuild ZIP.
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for file in OUT.rglob("*"):
        if file.is_file():
            zf.write(file, file.relative_to(OUT))

print("\nFINAL PACKAGE CREATED")
print("Folder:", OUT)
print("ZIP:", ZIP_PATH)
print("ZIP size (MB):", round(ZIP_PATH.stat().st_size / (1024**2), 2))
print("\nFinal four-section comparison:")
display(final_df.round(4))
print("\nStandalone Questionnaire MLP in final comparison: NO")
print("Number of thesis-facing rows:", len(final_df))
print("\nDownload asd_fyp_final_results.zip from Kaggle's Output panel.")


In [ ]:

# Cell 25 — Verify saved Late Fusion test results (NO TRAINING)

from pathlib import Path
import numpy as np
import json
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

FUSION = Path("/kaggle/working/asd_multimodal_prototype")
PRED_PATH = FUSION / "test_branch_predictions.npz"
RESULT_PATH = FUSION / "test_ablation_results.json"

if not PRED_PATH.exists():
    raise FileNotFoundError(f"Missing saved test predictions: {PRED_PATH}")

# Inspect saved array names first; don't assume their schema.
data = np.load(PRED_PATH, allow_pickle=False)

print("Saved arrays:", data.files)
for key in data.files:
    arr = data[key]
    print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")

print("\nSaved detailed results file exists:", RESULT_PATH.exists())
if RESULT_PATH.exists():
    saved = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
    print("\nSaved fusion evaluation record:")
    print(json.dumps(saved, indent=2)[:6000])


In [ ]:

# Cell 26 — Direct Late Fusion evaluation (NO TRAINING)

import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

data = np.load(
    "/kaggle/working/asd_multimodal_prototype/test_branch_predictions.npz"
)

p_img = data["p_img"].astype(float).ravel()
p_q = data["p_q"].astype(float).ravel()
y = data["y"].astype(int).ravel()

# Basic integrity checks
assert len(p_img) == len(p_q) == len(y), "Array lengths do not match."
assert set(np.unique(y)).issubset({0, 1}), "Labels must be binary."
assert np.isfinite(p_img).all() and np.isfinite(p_q).all()
assert np.isfinite(y).all()
assert ((p_img >= 0) & (p_img <= 1)).all(), "Image outputs are not probabilities."
assert ((p_q >= 0) & (p_q <= 1)).all(), "Questionnaire outputs are not probabilities."

# Equal-weight probability averaging
p_fusion = 0.5 * p_img + 0.5 * p_q
y_pred = (p_fusion >= 0.5).astype(int)

print("TEST EXAMPLES:", len(y))
print("Label counts:", dict(zip(*np.unique(y, return_counts=True))))

print("\n--- DIRECTLY RECALCULATED LATE FUSION ---")
print(f"Accuracy:  {accuracy_score(y, y_pred):.4f}")
print(f"Precision: {precision_score(y, y_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y, y_pred, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y, y_pred, zero_division=0):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y, p_fusion):.4f}")
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y, y_pred, labels=[0, 1]))

print("\nClassification report:")
print(classification_report(
    y, y_pred, labels=[0, 1],
    target_names=["Class 0", "Class 1"],
    zero_division=0
))

print("\n--- COMPARISON WITH IMAGE-ONLY ---")
img_pred = (p_img >= 0.5).astype(int)
print(f"Image-only accuracy: {accuracy_score(y, img_pred):.4f}")
print(f"Fusion accuracy:     {accuracy_score(y, y_pred):.4f}")
print(f"Image-only ROC-AUC:  {roc_auc_score(y, p_img):.4f}")
print(f"Fusion ROC-AUC:      {roc_auc_score(y, p_fusion):.4f}")

print("\nNote: This recalculation assumes p_img and p_q are probabilities")
print("for correctly aligned examples and that 50:50 averaging is the")
print("intended fusion rule. It does not establish participant identity.")


In [ ]:
# Branch Contribution and Fusion Diagnostic
# No training; uses saved test predictions only.

import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

data = np.load(
    "/kaggle/working/asd_multimodal_prototype/test_branch_predictions.npz"
)

p_img = data["p_img"].astype(float).ravel()
p_q   = data["p_q"].astype(float).ravel()
y     = data["y"].astype(int).ravel()

def evaluate(name, probs):
    pred = (probs >= 0.5).astype(int)
    print(f"\n{name}")
    print(f"Accuracy: {accuracy_score(y, pred):.4f}")
    print(f"ROC-AUC:  {roc_auc_score(y, probs):.4f}")
    print("Confusion matrix:")
    print(confusion_matrix(y, pred, labels=[0, 1]))

evaluate("1. Image branch only", p_img)
evaluate("2. Questionnaire branch only", p_q)
evaluate("3. Equal-weight fusion", 0.5 * p_img + 0.5 * p_q)

print("\n--- PREDICTION DIAGNOSTICS ---")
print("Image probability range:",
      round(p_img.min(), 4), "to", round(p_img.max(), 4))
print("Questionnaire probability range:",
      round(p_q.min(), 4), "to", round(p_q.max(), 4))

img_pred = (p_img >= 0.5).astype(int)
q_pred = (p_q >= 0.5).astype(int)
fusion_pred = ((0.5 * p_img + 0.5 * p_q) >= 0.5).astype(int)

print("Image/questionnaire predictions disagree:",
      int(np.sum(img_pred != q_pred)), "/", len(y))
print("Fusion predictions equal questionnaire predictions:",
      bool(np.array_equal(fusion_pred, q_pred)))
print("Fusion predictions equal image predictions:",
      bool(np.array_equal(fusion_pred, img_pred)))

In [ ]:
# Detailed Error Analysis (No Training)

import numpy as np

data = np.load(
    "/kaggle/working/asd_multimodal_prototype/test_branch_predictions.npz"
)

p_img = data["p_img"].astype(float).ravel()
p_q = data["p_q"].astype(float).ravel()
y = data["y"].astype(int).ravel()

p_fusion = 0.5 * p_img + 0.5 * p_q

img_pred = (p_img >= 0.5).astype(int)
q_pred = (p_q >= 0.5).astype(int)
fusion_pred = (p_fusion >= 0.5).astype(int)

img_wrong = img_pred != y
q_wrong = q_pred != y
fusion_wrong = fusion_pred != y

print("Image branch errors:", int(img_wrong.sum()))
print("Questionnaire branch errors:", int(q_wrong.sum()))
print("Fusion errors:", int(fusion_wrong.sum()))

print("\n--- FUSION VS QUESTIONNAIRE ---")
print("Fusion fixes questionnaire errors:",
      int(np.sum(q_wrong & ~fusion_wrong)))
print("Fusion introduces errors where questionnaire was correct:",
      int(np.sum(~q_wrong & fusion_wrong)))

print("\n--- FUSION VS IMAGE ---")
print("Fusion fixes image errors:",
      int(np.sum(img_wrong & ~fusion_wrong)))
print("Fusion introduces errors where image was correct:",
      int(np.sum(~img_wrong & fusion_wrong)))

print("\n--- DISAGREEMENT CASES ---")
idx = np.where(img_pred != q_pred)[0]
print("Total disagreements:", len(idx))
print("Index | True | Image prob | Questionnaire prob | Fusion prob | "
      "Image correct | Questionnaire correct | Fusion correct")

for i in idx[:30]:
    print(
        f"{i:5d} | {y[i]:4d} | {p_img[i]:10.4f} | "
        f"{p_q[i]:18.4f} | {p_fusion[i]:11.4f} | "
        f"{img_pred[i] == y[i]!s:13s} | "
        f"{q_pred[i] == y[i]!s:20s} | "
        f"{fusion_pred[i] == y[i]!s}"
    )

In [ ]:
# Questionnaire Leakage Audit (Read-only)

from pathlib import Path
import pandas as pd
import json

ROOTS = [
    Path("/kaggle/working"),
    Path("/kaggle/input/datasets/shahbaz18/backup/asd_care_results"),
    Path("/kaggle/input/datasets/ronakp004/autism-spectrum-detection-from-kaggle-zenodo"),
]

print("=" * 70)
print("1. QUESTIONNAIRE MODEL / PREPROCESSING ARTIFACTS")
print("=" * 70)

for root in ROOTS:
    if not root.exists():
        continue

    print(f"\nROOT: {root}")
    count = 0

    for pattern in [
        "*questionnaire*.keras",
        "*questionnaire*.json",
        "*scaler*.pkl",
        "*scaler*.joblib",
        "*preprocess*.json",
        "*feature*.json",
    ]:
        for path in root.rglob(pattern):
            print(path)
            count += 1
            if count >= 50:
                print("Output limited to 50 files for this root.")
                break
        if count >= 50:
            break

print("\n" + "=" * 70)
print("2. CSV FILES WITH DIAGNOSTIC / QUESTIONNAIRE COLUMNS")
print("=" * 70)

keywords = [
    "phenotyp", "question", "clinical", "autism",
    "asd", "behavior", "behaviour"
]

shown = 0
for root in ROOTS:
    if not root.exists():
        continue

    for path in root.rglob("*.csv"):
        if not any(k in path.name.lower() for k in keywords):
            continue

        try:
            df = pd.read_csv(path, nrows=5, encoding_errors="replace")
            print(f"\nFILE: {path}")
            print("Preview shape (first 5 rows):", df.shape)
            print("Columns:")
            print(df.columns.tolist())
            shown += 1
        except Exception as e:
            print(f"\nCould not read {path}: {e}")

        if shown >= 15:
            print("\nStopped after 15 CSV files.")
            break
    if shown >= 15:
        break

if shown == 0:
    print("No matching CSV files found in the listed roots.")

# 12. Resume Workflow and Fusion Packaging

Use these cells to locate saved components and package the existing fusion artifacts.

In [ ]:

# Resume Cell 1 — Check saved work (no training)

from pathlib import Path
import json
import numpy as np

backup = Path(
    "/kaggle/input/datasets/shahbaz18/backup/asd_care_results"
)

fusion_dir = Path(
    "/kaggle/working/asd_multimodal_prototype"
)

resume_dir = Path("/kaggle/working/asd_resume")

paths = {
    "MobileNetV2 model": backup / "mobilenetv2_best.keras",
    "EfficientNetB0 model": backup / "efficientnetb0_best.keras",
    "ResNet50 model": backup / "resnet50_best.keras",
    "Questionnaire model": backup / "questionnaire_mlp_best.keras",
    "Saved fusion predictions": fusion_dir / "test_branch_predictions.npz",
    "Saved ablation results": fusion_dir / "test_ablation_results.json",
    "Resume archive": resume_dir,
}

print("=== SAVED ARTIFACT CHECK ===")
for name, path in paths.items():
    print(f"{name}: {'FOUND' if path.exists() else 'NOT FOUND'}")
    if path.is_file():
        print(f"  Size: {path.stat().st_size / (1024**2):.2f} MB")

print("\n=== CURRENT KERNEL ===")
print("NumPy:", np.__version__)

try:
    import tensorflow as tf
    print("TensorFlow:", tf.__version__)
    print("Available GPUs:", tf.config.list_physical_devices("GPU"))
except Exception as e:
    print("TensorFlow check:", repr(e))

pred_path = fusion_dir / "test_branch_predictions.npz"
if pred_path.exists():
    with np.load(pred_path, allow_pickle=False) as data:
        print("\nSaved prediction arrays:", data.files)
        for key in data.files:
            print(f"{key}: shape={data[key].shape}")


In [ ]:

# Resume Cell 2 — Locate questionnaire data and feature files

from pathlib import Path
import pandas as pd

search_roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input/datasets/shahbaz18/backup/asd_care_results"),
    Path("/kaggle/input/datasets/ronakp004/autism-spectrum-detection-from-kaggle-zenodo"),
]

keywords = (
    "question", "phenotyp", "clinical", "feature",
    "scaler", "preprocess", "fusion", "ablation"
)

seen = set()
count = 0

for root in search_roots:
    if not root.exists():
        continue

    print(f"\n=== SEARCHING: {root} ===")

    for path in root.rglob("*"):
        if not path.is_file() or path in seen:
            continue

        seen.add(path)

        if any(k in path.name.lower() for k in keywords):
            print(path)
            count += 1

            if path.suffix.lower() == ".csv":
                try:
                    df = pd.read_csv(path, nrows=3, encoding_errors="replace")
                    print("  Columns:", df.columns.tolist())
                    print("  Preview shape:", df.shape)
                except Exception as e:
                    print("  CSV read error:", str(e)[:200])

            if count >= 100:
                print("\nSearch output limited to 100 files.")
                break

    if count >= 100:
        break

print(f"\nTotal matching files listed: {count}")


In [ ]:

# Resume Cell 3 — Inspect saved questionnaire artifacts

from pathlib import Path
import json
import numpy as np

B = Path("/kaggle/input/datasets/shahbaz18/backup/asd_care_results")

files = [
    "questionnaire_preprocessing.json",
    "questionnaire_scaler.json",
    "questionnaire_mlp_test_results.json",
    "questionnaire_mlp_history.json",
]

for name in files:
    p = B / name
    print(f"\n{'='*25} {name} {'='*25}")

    if not p.exists():
        print("NOT FOUND")
        continue

    try:
        obj = json.loads(p.read_text(encoding="utf-8"))
        print(json.dumps(obj, indent=2)[:12000])
    except Exception as e:
        print("Read error:", repr(e))

pred_path = B / "questionnaire_mlp_test_predictions.npz"
print(f"\n{'='*25} TEST PREDICTIONS {'='*25}")

if pred_path.exists():
    with np.load(pred_path, allow_pickle=False) as z:
        print("Arrays:", z.files)
        for k in z.files:
            a = z[k]
            print(f"\n{k}: shape={a.shape}, dtype={a.dtype}")
            print("First 10 values:", a.reshape(-1)[:10])
else:
    print("Test predictions file not found.")


In [ ]:

# Cell 30 — Find original questionnaire data and training code
# Read-only: no training and no file changes.

from pathlib import Path

roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

extensions = {".csv", ".ipynb", ".py", ".parquet", ".xlsx"}
keywords = [
    "autism", "question", "screen", "phenotyp",
    "clinical", "fusion", "asd", "train"
]

found = set()

for root in roots:
    if not root.exists():
        continue

    print(f"\n=== SEARCHING {root} ===")

    for p in root.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in extensions:
            continue

        name = p.name.lower()
        if any(k in name for k in keywords):
            found.add(p)

for p in sorted(found, key=str):
    print(p)

print(f"\nTotal matching files: {len(found)}")


In [ ]:

from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset/Dataset_csv"
)

print("Directory exists:", DATA_DIR.exists())

if DATA_DIR.exists():
    print("\nFiles:")
    for p in sorted(DATA_DIR.iterdir()):
        if p.is_file():
            print(f"{p.name} ({p.stat().st_size / 1024:.1f} KB)")

    for p in sorted(DATA_DIR.glob("*.csv")):
        print(f"\n{'=' * 60}\nFILE: {p.name}")
        try:
            df = pd.read_csv(p, encoding_errors="replace")
            print("Shape:", df.shape)
            print("Columns:", df.columns.tolist())
            print("\nFirst 5 rows:")
            print(df.head().to_string(max_cols=20))

            for c in df.columns:
                if any(k in c.lower() for k in [
                    "class", "label", "target", "asd", "autism",
                    "diagnosis", "result", "participant", "subject", "id"
                ]):
                    print(f"\nPotential target/ID: {c}")
                    print(df[c].value_counts(dropna=False).head(10).to_dict())
        except Exception as e:
            print("Read error:", repr(e))
else:
    print("Dataset directory not found.")


In [ ]:

# Cell 32 — Questionnaire and split integrity audit
from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    "/kaggle/input/datasets/ajithdari/"
    "autism-spectrum-disorder-multi-modal-dataset/Dataset_csv"
)

files = {
    split: pd.read_csv(DATA_DIR / f"{split}.csv", encoding_errors="replace")
    for split in ["train", "valid", "test"]
}

feature_cols = [
    "A1", "A2", "A3", "A4", "A5",
    "A6", "A7", "A8", "A9", "A10",
    "Age", "Sex", "Jauundice", "Family_ASD"
]

all_df = pd.concat(
    [df.assign(_split=name) for name, df in files.items()],
    ignore_index=True
)

print("=" * 65)
print("1. MISSING VALUES")
print("=" * 65)
print(all_df[feature_cols + ["Class", "image_path"]].isna().sum())

print("\n" + "=" * 65)
print("2. DUPLICATE QUESTIONNAIRE PATTERNS")
print("=" * 65)

# Does the exact same questionnaire pattern occur many times?
all_df["_questionnaire_key"] = (
    all_df[feature_cols].astype(str).agg("|".join, axis=1)
)

print("Unique questionnaire patterns:", all_df["_questionnaire_key"].nunique())
print("Rows belonging to repeated patterns:",
      int(all_df["_questionnaire_key"].duplicated(keep=False).sum()))

print("\nMost repeated patterns with class/split breakdown:")
repeated_keys = (
    all_df["_questionnaire_key"]
    .value_counts()
    .loc[lambda s: s > 1]
    .head(5)
    .index
)

for key in repeated_keys:
    subset = all_df[all_df["_questionnaire_key"] == key]
    print("\nPattern count:", len(subset))
    print("Classes:", subset["Class"].value_counts().to_dict())
    print("Splits:", subset["_split"].value_counts().to_dict())
    print("Example image paths:", subset["image_path"].head(4).tolist())

print("\n" + "=" * 65)
print("3. QUESTIONNAIRE FEATURES VS TARGET")
print("=" * 65)

for col in feature_cols:
    print(f"\n{col}:")
    print(pd.crosstab(all_df[col], all_df["Class"], dropna=False).to_string())

print("\n" + "=" * 65)
print("4. IMAGE PATH / CLASS CONSISTENCY")
print("=" * 65)

all_df["_path_class"] = all_df["image_path"].astype(str).str.lower().apply(
    lambda p: "YES" if "/autistic/" in p else (
        "NO" if "/non_autistic/" in p or "/non-autistic/" in p
        or "/typical/" in p else "UNKNOWN"
    )
)

print(pd.crosstab(all_df["_path_class"], all_df["Class"], dropna=False))

print("\n" + "=" * 65)
print("5. IMAGE PATH OVERLAP BETWEEN SPLITS")
print("=" * 65)

for a, b in [("train", "valid"), ("train", "test"), ("valid", "test")]:
    paths_a = set(files[a]["image_path"].astype(str))
    paths_b = set(files[b]["image_path"].astype(str))
    print(f"{a} vs {b}: exact shared paths =", len(paths_a & paths_b))

print("\n" + "=" * 65)
print("6. CLASS COUNTS WITHIN EACH SPLIT")
print("=" * 65)

for split, df in files.items():
    print(split, df["Class"].value_counts(dropna=False).to_dict())


In [ ]:

# Cell: Locate Saved Fusion Components
# Safe to run — no training.

import os

candidates = {
    "ResNet50": [
        "/kaggle/working/asd_fyp_final_results/models/resnet50_best.keras",
        "/kaggle/input/datasets/shahbaz18/backup/asd_care_results/resnet50_best.keras",
    ],
    "Questionnaire MLP": [
        "/kaggle/working/asd_fyp_final_results/models/questionnaire_mlp_best.keras",
        "/kaggle/input/datasets/shahbaz18/backup/asd_care_results/questionnaire_mlp_best.keras",
    ],
}

for name, paths in candidates.items():
    print(f"\n{name}:")
    for path in paths:
        print(" ", os.path.isfile(path), path)

print("\nPrediction files:")
for root in [
    "/kaggle/working/asd_fyp_final_results",
    "/kaggle/input/datasets/shahbaz18/backup/asd_care_results",
]:
    if os.path.isdir(root):
        for folder, _, files in os.walk(root):
            for filename in files:
                if filename.endswith(".npz") and any(
                    word in filename.lower()
                    for word in ["prediction", "branch", "test"]
                ):
                    print(os.path.join(folder, filename))


In [ ]:

# Cell 2: Package the Late-Fusion Model
# Does not retrain either model.

import os
import json
import shutil
import zipfile

BASE = "/kaggle/working/asd_fyp_final_results"
OUT = "/kaggle/working/asd_fusion_standalone"
ZIP_PATH = "/kaggle/working/asd_fusion_standalone.zip"
BACKUP = "/kaggle/input/datasets/shahbaz18/backup/asd_care_results"

os.makedirs(OUT, exist_ok=True)
os.makedirs(f"{OUT}/models", exist_ok=True)
os.makedirs(f"{OUT}/fusion_results", exist_ok=True)

# Copy trained model components.
model_sources = {
    "resnet50_best.keras": [
        f"{BASE}/models/resnet50_best.keras",
        f"{BACKUP}/resnet50_best.keras",
    ],
    "questionnaire_mlp_best.keras": [
        f"{BASE}/models/questionnaire_mlp_best.keras",
        f"{BACKUP}/questionnaire_mlp_best.keras",
    ],
}

for filename, candidates in model_sources.items():
    source = next((p for p in candidates if os.path.isfile(p)), None)
    if source is None:
        raise FileNotFoundError(f"Missing required model: {filename}")
    shutil.copy2(source, f"{OUT}/models/{filename}")
    print("Saved model:", filename)

# Copy preprocessing/scaler files if present.
for filename in [
    "questionnaire_preprocessing.json",
    "questionnaire_scaler.json",
]:
    candidates = [
        f"{BASE}/{filename}",
        f"{BACKUP}/{filename}",
    ]
    source = next((p for p in candidates if os.path.isfile(p)), None)
    if source:
        shutil.copy2(source, f"{OUT}/{filename}")
        print("Saved preprocessing:", filename)
    else:
        print("WARNING: preprocessing file not found:", filename)

# Copy saved branch predictions.
for filename in [
    "train_branch_predictions.npz",
    "valid_branch_predictions.npz",
    "test_branch_predictions.npz",
]:
    source = f"{BASE}/fusion_results/{filename}"
    if os.path.isfile(source):
        shutil.copy2(source, f"{OUT}/fusion_results/{filename}")
        print("Saved predictions:", filename)
    else:
        print("WARNING: missing prediction file:", filename)

# Save fusion definition and limitations.
config = {
    "method": "Late fusion by probability averaging",
    "image_model": "ResNet50",
    "questionnaire_model": "Questionnaire MLP",
    "image_weight": 0.5,
    "questionnaire_weight": 0.5,
    "classification_threshold": 0.5,
    "recorded_test_accuracy": 0.985,
    "accuracy_unit": "fraction",
    "protocol_note": (
        "Recorded result under a constructed synthetic-pairing protocol. "
        "Not evidence of performance on genuinely paired participants "
        "and not clinically validated."
    ),
}
with open(f"{OUT}/fusion_config.json", "w") as f:
    json.dump(config, f, indent=2)

# Save the fusion calculation for reuse.
fusion_code = '''import numpy as np

def fuse_probabilities(image_probabilities,
                       questionnaire_probabilities,
                       image_weight=0.5,
                       questionnaire_weight=0.5,
                       threshold=0.5):
    """
    Inputs must be aligned probabilities for the same ordered examples.
    Returns fused probabilities and binary predictions.
    """
    image_probabilities = np.asarray(image_probabilities).reshape(-1)
    questionnaire_probabilities = np.asarray(
        questionnaire_probabilities
    ).reshape(-1)

    if image_probabilities.shape != questionnaire_probabilities.shape:
        raise ValueError("Both branches must have the same number of examples.")

    if not np.isclose(image_weight + questionnaire_weight, 1.0):
        raise ValueError("Fusion weights must sum to 1.")

    fused = (
        image_weight * image_probabilities
        + questionnaire_weight * questionnaire_probabilities
    )
    predicted_labels = (fused >= threshold).astype(int)
    return fused, predicted_labels
'''
with open(f"{OUT}/fusion.py", "w") as f:
    f.write(fusion_code)

# Write a manifest.
manifest = []
for root, _, files in os.walk(OUT):
    for filename in files:
        path = os.path.join(root, filename)
        manifest.append({
            "file": os.path.relpath(path, OUT),
            "size_bytes": os.path.getsize(path)
        })

with open(f"{OUT}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

# Build ZIP from the standalone folder.
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUT):
        for filename in files:
            full = os.path.join(root, filename)
            zf.write(full, os.path.relpath(full, OUT))

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    assert zf.testzip() is None, "ZIP integrity check failed"

print("\nFusion package created:", ZIP_PATH)
print(f"ZIP size: {os.path.getsize(ZIP_PATH)/(1024**2):.2f} MB")
print("ZIP integrity: PASS")


# 13. Publication Figures and Phase 1 Outputs

Create figures from saved predictions and histories. This section should not train models.

In [ ]:

# CELL 1 — AUDIT SAVED RESULTS
# Safe to run: no model training and no files modified.

import os

BASE = "/kaggle/working/asd_fyp_final_results"
BACKUP = "/kaggle/input/datasets/shahbaz18/backup/asd_care_results"

print("Results folder exists:", os.path.isdir(BASE))
print("Backup folder exists:", os.path.isdir(BACKUP))

KEYWORDS = (
    "history", "prediction", "branch", "result", "metric",
    "comparison", "confusion", "roc", "fusion", "json", "csv", "npz"
)

for folder in [BASE, BACKUP]:
    if not os.path.isdir(folder):
        continue

    print(f"\n{'=' * 25}\nFILES IN: {folder}\n{'=' * 25}")

    for root, dirs, files in os.walk(folder):
        for filename in sorted(files):
            if any(word in filename.lower() for word in KEYWORDS):
                path = os.path.join(root, filename)
                size_mb = os.path.getsize(path) / (1024 ** 2)
                print(f"{os.path.relpath(path, folder)} | {size_mb:.2f} MB")


In [ ]:

# CELL 2 — INSPECT PREDICTION ARRAYS AND METRICS
# Read-only. Does not train models or change files.

import os
import json

import numpy as np

BASE = "/kaggle/working/asd_fyp_final_results"

files = [
    "mobilenetv2_test_predictions.npz",
    "efficientnetb0_test_predictions.npz",
    "resnet50_test_predictions.npz",
    "questionnaire_mlp_test_predictions.npz",
    "fusion_results/test_branch_predictions.npz",
]

for filename in files:
    path = os.path.join(BASE, filename)
    print(f"\n{'=' * 65}\n{filename}\n{'=' * 65}")

    if not os.path.isfile(path):
        print("MISSING")
        continue

    with np.load(path, allow_pickle=False) as data:
        for key in data.files:
            arr = data[key]
            print(
                f"{key}: shape={arr.shape}, dtype={arr.dtype}",
                f", sample={arr.reshape(-1)[:5]}"
                if arr.size and arr.dtype.kind in "biuf" else ""
            )

for filename in [
    "final_four_section_results.json",
    "final_model_comparison.json",
    "fusion_results/test_ablation_results.json",
    "mobilenetv2_history.json",
    "efficientnetb0_history.json",
    "resnet50_history.json",
]:
    path = os.path.join(BASE, filename)
    print(f"\n{'=' * 65}\n{filename}\n{'=' * 65}")

    if os.path.isfile(path):
        with open(path, "r") as f:
            content = json.load(f)
        print(json.dumps(content, indent=2)[:5000])
    else:
        print("MISSING")


In [ ]:

# CELL 3 — PUBLICATION FIGURES: TEST PERFORMANCE
# No model training. Uses saved test predictions.

import os
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve,
    precision_recall_curve, average_precision_score
)

BASE = "/kaggle/working/asd_fyp_final_results"
OUT = "/kaggle/working/asd_graphs"
os.makedirs(OUT, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "figure.titlesize": 15,
})

def load_npz(relative_path):
    path = os.path.join(BASE, relative_path)
    if not os.path.isfile(path):
        raise FileNotFoundError(path)
    with np.load(path, allow_pickle=False) as d:
        return {k: d[k].copy() for k in d.files}

def save_figure(fig, name):
    fig.savefig(os.path.join(OUT, name + ".png"),
                dpi=300, bbox_inches="tight")
    fig.savefig(os.path.join(OUT, name + ".pdf"),
                bbox_inches="tight")
    plt.show()
    plt.close(fig)

# Load individual image-model predictions.
files = {
    "MobileNetV2": "mobilenetv2_test_predictions.npz",
    "EfficientNetB0": "efficientnetb0_test_predictions.npz",
    "ResNet50": "resnet50_test_predictions.npz",
}

models = {}
for name, filename in files.items():
    d = load_npz(filename)
    models[name] = {
        "y": d["y_true"].astype(int).ravel(),
        "p": d["y_prob"].astype(float).ravel(),
        "pred": d["y_pred"].astype(int).ravel(),
    }

# Reconstruct fusion from its saved branch probabilities.
fd = load_npz("fusion_results/test_branch_predictions.npz")
fusion_y = fd["y"].astype(int).ravel()
p_img = fd["p_img"].astype(float).ravel()
p_q = fd["p_q"].astype(float).ravel()

if not (len(fusion_y) == len(p_img) == len(p_q)):
    raise ValueError("Fusion arrays have different lengths.")

if not np.array_equal(fusion_y, models["ResNet50"]["y"]):
    print("WARNING: Fusion and standalone ResNet50 label ordering differs.")
    print("Fusion results will use the labels saved in the fusion file.")

fusion_p = 0.5 * p_img + 0.5 * p_q
fusion_pred = (fusion_p >= 0.5).astype(int)

models["Late Fusion (50:50)"] = {
    "y": fusion_y,
    "p": fusion_p,
    "pred": fusion_pred,
}

# Verify probability ranges and calculate metrics from predictions.
rows = []
for name, d in models.items():
    y, p, pred = d["y"], d["p"], d["pred"]

    if len(y) != len(p) or len(y) != len(pred):
        raise ValueError(f"Array length mismatch for {name}.")
    if not np.isin(y, [0, 1]).all():
        raise ValueError(f"Unexpected labels for {name}.")
    if np.any((p < 0) | (p > 1)):
        raise ValueError(f"Probabilities outside [0, 1] for {name}.")

    rows.append({
        "model": name,
        "test_samples": len(y),
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1_score": f1_score(y, pred, zero_division=0),
        "roc_auc": roc_auc_score(y, p),
        "average_precision": average_precision_score(y, p),
    })

metrics = pd.DataFrame(rows)
metrics.to_csv(os.path.join(OUT, "recomputed_test_metrics.csv"), index=False)

print("RECOMPUTED TEST METRICS")
display(metrics.round(4))

# Figure 1: Main model comparison.
metric_cols = ["accuracy", "precision", "recall", "f1_score", "roc_auc"]
plot_df = metrics.set_index("model")[metric_cols]

fig, ax = plt.subplots(figsize=(12, 6))
plot_df.plot(kind="bar", ax=ax)
ax.set_title("Test Performance Comparison")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.legend(title="Metric", loc="lower right")
ax.tick_params(axis="x", rotation=15)
save_figure(fig, "01_test_performance_comparison")

# Figure 2: Confusion-matrix heatmaps.
for i, (name, d) in enumerate(models.items(), start=2):
    cm = confusion_matrix(d["y"], d["pred"], labels=[0, 1])

    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(f"Confusion Matrix — {name}")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks([0, 1], labels=["Non-ASD (0)", "ASD (1)"])
    ax.set_yticks([0, 1], labels=["Non-ASD (0)", "ASD (1)"])

    for r in range(2):
        for c in range(2):
            ax.text(c, r, str(cm[r, c]),
                    ha="center", va="center", fontsize=13)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    save_figure(fig, f"{i:02d}_confusion_matrix_{name.lower().replace(' ', '_').replace('(', '').replace(')', '').replace(':', '')}")

# Figure 6: Combined ROC curves.
fig, ax = plt.subplots(figsize=(7.5, 6))
for name, d in models.items():
    fpr, tpr, _ = roc_curve(d["y"], d["p"])
    auc = roc_auc_score(d["y"], d["p"])
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

ax.plot([0, 1], [0, 1], linestyle="--", label="Chance")
ax.set_title("ROC Curves on the Test Set")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.03)
ax.grid(alpha=0.3)
ax.legend(loc="lower right", fontsize=9)
save_figure(fig, "06_roc_curves")

# Figure 7: Combined precision-recall curves.
fig, ax = plt.subplots(figsize=(7.5, 6))
for name, d in models.items():
    precision, recall, _ = precision_recall_curve(d["y"], d["p"])
    ap = average_precision_score(d["y"], d["p"])
    ax.plot(recall, precision, label=f"{name} (AP={ap:.3f})")

ax.set_title("Precision–Recall Curves on the Test Set")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.03)
ax.grid(alpha=0.3)
ax.legend(loc="lower left", fontsize=9)
save_figure(fig, "07_precision_recall_curves")

# Save fusion details for reproducibility.
np.savez_compressed(
    os.path.join(OUT, "reconstructed_fusion_test_predictions.npz"),
    y_true=fusion_y,
    p_image=p_img,
    p_questionnaire=p_q,
    p_fusion=fusion_p,
    y_pred=fusion_pred,
)

with open(os.path.join(OUT, "figure_notes.json"), "w") as f:
    json.dump({
        "fusion_rule": "p_fusion = 0.5 * p_img + 0.5 * p_q",
        "classification_threshold": 0.5,
        "fusion_predictions_source":
            "Saved fusion_results/test_branch_predictions.npz",
        "caution":
            "Synthetic pairing limitation applies. These figures do not "
            "establish clinical validity or genuine participant-level "
            "multimodal performance.",
        "metrics_source":
            "Recomputed from saved test predictions in this cell."
    }, f, indent=2)

# Package all generated outputs.
zip_path = "/kaggle/working/asd_graphs_phase1.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, filenames in os.walk(OUT):
        for filename in filenames:
            full = os.path.join(root, filename)
            zf.write(full, os.path.relpath(full, OUT))

with zipfile.ZipFile(zip_path, "r") as zf:
    assert zf.testzip() is None

print("\nFigures saved to:", OUT)
print("Download package:", zip_path)
print("ZIP integrity: PASS")


In [ ]:

# Cell: Phase 1 — Generate Training-History Graphs
# No model retraining. Saves PNG/PDF figures and updates the ZIP.

import os, json, zipfile
import matplotlib.pyplot as plt

BASE = "/kaggle/working/asd_fyp_final_results"
OUT = "/kaggle/working/asd_graphs"
os.makedirs(OUT, exist_ok=True)

models = {
    "MobileNetV2": "mobilenetv2_history.json",
    "EfficientNetB0": "efficientnetb0_history.json",
    "ResNet50": "resnet50_history.json",
    "Questionnaire MLP": "questionnaire_mlp_history.json",
}

histories = {}

for name, filename in models.items():
    path = os.path.join(BASE, filename)
    if not os.path.isfile(path):
        print(f"MISSING: {name}: {path}")
        continue

    with open(path, "r") as f:
        data = json.load(f)

    # Accept either a direct Keras History dictionary or a wrapped dictionary.
    if isinstance(data, dict) and "history" in data and isinstance(data["history"], dict):
        data = data["history"]

    if isinstance(data, dict):
        histories[name] = data
        print(f"Loaded {name}: {len(data.get('loss', []))} epochs")
    else:
        print(f"UNEXPECTED FORMAT: {name}: {type(data)}")

if not histories:
    raise FileNotFoundError(
        "No history files loaded. Send me the output above; do not retrain."
    )

# Plot each metric only when at least one model has it.
metric_specs = [
    ("accuracy", "Accuracy", "accuracy"),
    ("loss", "Loss", "loss"),
    ("auc", "AUC", "auc"),
]

for key, title, ylabel in metric_specs:
    available = [
        (name, h) for name, h in histories.items()
        if key in h and len(h[key]) > 0
    ]
    if not available:
        print(f"Skipped {title}: no saved training values.")
        continue

    fig, ax = plt.subplots(figsize=(9, 5.5))

    for name, h in available:
        ax.plot(range(1, len(h[key]) + 1), h[key],
                marker="o", markersize=3, label=f"{name} train")

        val_key = f"val_{key}"
        if val_key in h and len(h[val_key]) > 0:
            ax.plot(range(1, len(h[val_key]) + 1), h[val_key],
                    marker=".", linestyle="--", label=f"{name} validation")

    ax.set_title(f"Training and Validation {title}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()

    stem = f"training_history_{key}"
    fig.savefig(os.path.join(OUT, stem + ".png"), dpi=300, bbox_inches="tight")
    fig.savefig(os.path.join(OUT, stem + ".pdf"), bbox_inches="tight")
    plt.show()
    plt.close(fig)

# Save a record of exactly which histories were available.
summary = {
    name: {
        metric: len(values)
        for metric, values in history.items()
        if isinstance(values, list)
    }
    for name, history in histories.items()
}
with open(os.path.join(OUT, "training_history_manifest.json"), "w") as f:
    json.dump(summary, f, indent=2)

# Rebuild the ZIP so it includes both the previous figures and new history graphs.
zip_path = "/kaggle/working/asd_graphs_phase1.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for filename in files:
            path = os.path.join(root, filename)
            z.write(path, os.path.relpath(path, OUT))

print("\nSaved graph directory:", OUT)
print("Updated ZIP:", zip_path)
print("History files loaded:", list(histories.keys()))
print("ZIP size (MB):", round(os.path.getsize(zip_path) / (1024**2), 2))
